# NimbusPay Ticket Triage – Fine-Tuning a Small Language Model

## Project Overview

This project fine-tunes an open-weight language model to classify NimbusPay customer support tickets and generate structured JSON triage records.

**Base model:** Qwen2.5-1.5B-Instruct  
**Fine-tuning approach:** 4-bit Quantized Low-Rank Adaptation (QLoRA)  
**Training dataset:** 1,445 audited and cleaned records  
**Development dataset:** 200 labelled records  
**Test dataset:** 400 unlabelled records  
**Hardware:** Google Colab NVIDIA T4 GPU

The objective is to teach the model to identify ticket categories, determine priority, extract transaction information, identify language, and decide whether human support is required.

The notebook covers data auditing, baseline evaluation, parameter-efficient fine-tuning, development-set evaluation, error analysis, and test prediction generation.

> **Notebook maintenance note:** This organised copy preserves the original code and saved outputs. A few repeated upload, model-loading, and troubleshooting cells are retained for now to avoid deleting work. Before claiming a fresh-runtime top-to-bottom run, consolidate those cells and verify the notebook from a clean T4 runtime.

## 2. Data Loading and Initial Inspection

Load the supplied training data and inspect record counts, IDs, and message structure.

In [ ]:
from google.colab import files

uploaded = files.upload()

Saving train.jsonl to train.jsonl


In [ ]:
TRAIN_PATH = "/content/train.jsonl"

import os
print("File exists:", os.path.exists(TRAIN_PATH))

File exists: True


In [ ]:
import json
import pandas as pd
from pathlib import Path

TRAIN_PATH = "/content/train.jsonl"

with open(TRAIN_PATH, "r", encoding="utf-8") as f:
    train_rows = [json.loads(line) for line in f if line.strip()]

print("Total training rows:", len(train_rows))

print("\nFirst training example:")
print(json.dumps(train_rows[0], indent=2, ensure_ascii=False))

Total training rows: 1445

First training example:
{
  "id": "tr-00001",
  "messages": [
    {
      "role": "system",
      "content": "You are NimbusPay's ticket triage model. Read the ticket and return the triage JSON."
    },
    {
      "role": "user",
      "content": "Received: 2026-07-25 (Sat)\nSource: app-chat\n\nHi, I have a top up that didn't go through. Why? It was today. Mode: internet banking. My phone is a Redmi Note 12. It is 4.7k in total. My account balance is 258 rupees right now. The txn id is NP1599046332. Waiting for your reply."
    },
    {
      "role": "assistant",
      "content": "{\"category\": \"payment_failed\", \"priority\": \"P3\", \"amount_paise\": 470000, \"txn_date\": \"2026-07-25\", \"txn_id\": \"NP1599046332\", \"channel\": \"netbanking\", \"language\": \"en\", \"needs_human\": false}"
    }
  ]
}


In [ ]:
print("Unique IDs:", len({row["id"] for row in train_rows}))

print("\nMessage roles:")
print(pd.Series([
    msg["role"]
    for row in train_rows
    for msg in row["messages"]
]).value_counts())

print("\nFirst 5 assistant labels:")
for row in train_rows[:5]:
    print("\nID:", row["id"])
    print(row["messages"][-1]["content"])

Unique IDs: 1445

Message roles:
system       1445
user         1445
assistant    1445
Name: count, dtype: int64

First 5 assistant labels:

ID: tr-00001
{"category": "payment_failed", "priority": "P3", "amount_paise": 470000, "txn_date": "2026-07-25", "txn_id": "NP1599046332", "channel": "netbanking", "language": "en", "needs_human": false}

ID: tr-00002
{"category": "account_access", "priority": "P2", "amount_paise": null, "txn_date": null, "txn_id": null, "channel": null, "language": "en", "needs_human": false}

ID: tr-00003
{"category": "refund", "priority": "P2", "amount_paise": 1915100, "txn_date": "2026-09-02", "txn_id": "NP2416305516", "channel": null, "language": "hi-en", "needs_human": false}

ID: tr-00004
{"category": "refund", "priority": "P1", "amount_paise": 7892400, "txn_date": "2026-08-08", "txn_id": "NP5367592160", "channel": null, "language": "en", "needs_human": true}

ID: tr-00005
{"category": "account_access", "priority": "P2", "amount_paise": null, "txn_date": nul

## 3. Training Data Audit

Check label JSON structure, required keys, allowed values, and suspicious labels.

In [ ]:
import json
from collections import Counter

EXPECTED_KEYS = {
    "category",
    "priority",
    "amount_paise",
    "txn_date",
    "txn_id",
    "channel",
    "language",
    "needs_human"
}

invalid_json = []
wrong_keys = []
category_counts = Counter()
priority_counts = Counter()

for row in train_rows:
    row_id = row["id"]
    label = row["messages"][-1]["content"]

    try:
        data = json.loads(label)

        if not isinstance(data, dict):
            invalid_json.append(row_id)
            continue

        if set(data.keys()) != EXPECTED_KEYS:
            wrong_keys.append((row_id, list(data.keys())))

        if "category" in data:
            category_counts[data["category"]] += 1

        if "priority" in data:
            priority_counts[data["priority"]] += 1

    except json.JSONDecodeError:
        invalid_json.append(row_id)

print("Invalid JSON labels:", len(invalid_json))
print("Wrong or missing keys:", len(wrong_keys))

print("\nInvalid JSON IDs:")
print(invalid_json)

print("\nWrong key examples:")
for item in wrong_keys[:10]:
    print(item)

print("\nCategory distribution:")
print(category_counts)

print("\nPriority distribution:")
print(priority_counts)

Invalid JSON labels: 25
Wrong or missing keys: 70

Invalid JSON IDs:
['tr-00060', 'tr-00095', 'tr-00125', 'tr-00172', 'tr-00267', 'tr-00357', 'tr-00478', 'tr-00497', 'tr-00558', 'tr-00605', 'tr-00674', 'tr-00735', 'tr-00765', 'tr-00809', 'tr-00824', 'tr-00846', 'tr-00958', 'tr-00960', 'tr-00975', 'tr-00978', 'tr-00979', 'tr-00980', 'tr-01070', 'tr-01087', 'tr-01199']

Wrong key examples:
('tr-00020', ['category', 'prio', 'amount_paise', 'txn_date', 'txn_id', 'channel', 'language', 'needs_human'])
('tr-00049', ['category', 'prio', 'amount_paise', 'txn_date', 'txn_id', 'channel', 'language', 'needs_human'])
('tr-00058', ['category', 'prio', 'amount_paise', 'txn_date', 'txn_id', 'channel', 'language', 'needs_human'])
('tr-00072', ['category', 'prio', 'amount_paise', 'txn_date', 'txn_id', 'channel', 'language', 'needs_human'])
('tr-00077', ['category', 'prio', 'amount_paise', 'txn_date', 'txn_id', 'channel', 'language', 'needs_human'])
('tr-00114', ['category', 'prio', 'amount_paise', 'txn

In [ ]:
import json

# Inspect 5 records with the wrong key
wrong_key_ids = [item[0] for item in wrong_keys]

print("========== WRONG KEY EXAMPLES ==========")

for row in train_rows:
    if row["id"] in wrong_key_ids[:5]:
        print("\nID:", row["id"])
        print("TICKET:")
        print(row["messages"][1]["content"])
        print("LABEL:")
        print(row["messages"][-1]["content"])


# Inspect 5 records with invalid JSON
print("\n========== INVALID JSON EXAMPLES ==========")

for row in train_rows:
    if row["id"] in invalid_json[:5]:
        print("\nID:", row["id"])
        print("TICKET:")
        print(row["messages"][1]["content"])
        print("LABEL:")
        print(row["messages"][-1]["content"])

========== WRONG KEY EXAMPLES ==========

ID: tr-00020
TICKET:
Received: 2026-09-05 (Sat)
Source: ivr-transcript

[IVR transcript] Hey, How can I check if I have received a refund if it is not yet on my statement? 978 rupees is the amount. I have contacted support about this before. Mode: wallet. Using the Android app. This happened on 02-09-26. Thanks.
LABEL:
{"category": "refund", "prio": "low", "amount_paise": 97800, "txn_date": "02/09/2026", "txn_id": null, "channel": "wallet", "language": "en", "needs_human": true}

ID: tr-00049
TICKET:
Received: 2026-09-01 (Tue)
Source: ivr-transcript

[IVR transcript]  Regsitered email id kaise change karun? Dhanyavaad.
LABEL:
{"category": "other", "prio": "low", "amount_paise": null, "txn_date": null, "txn_id": null, "channel": null, "language": "hi-en", "needs_human": false}

ID: tr-00058
TICKET:
Received: 2026-09-13 (Sun)
Source: ivr-transcript

[IVR transcript] Dear NimbusPay support, I tried to make a transfer and it didn't work. This happe

In [ ]:
from collections import Counter

prio_values = Counter()
category_prio = Counter()

for row in train_rows:
    label = row["messages"][-1]["content"]

    try:
        data = json.loads(label)

        if "prio" in data:
            prio_values[data["prio"]] += 1
            category_prio[(data.get("category"), data["prio"])] += 1

    except json.JSONDecodeError:
        pass

print("Priority values in incorrect labels:")
print(prio_values)

print("\nCategory and priority combinations:")
for key, count in sorted(category_prio.items()):
    print(key, ":", count)

Priority values in incorrect labels:
Counter({'low': 44, 'medium': 15, 'high': 11})

Category and priority combinations:
('account_access', 'high') : 1
('account_access', 'medium') : 8
('fraud', 'high') : 3
('kyc', 'low') : 7
('offers', 'low') : 7
('other', 'low') : 9
('payment_failed', 'high') : 3
('payment_failed', 'low') : 10
('payment_failed', 'medium') : 4
('refund', 'high') : 4
('refund', 'low') : 11
('refund', 'medium') : 3


In [ ]:
print("========== ALL 25 INVALID LABELS ==========")

for row in train_rows:
    label = row["messages"][-1]["content"]

    try:
        json.loads(label)
    except json.JSONDecodeError:
        print("\nID:", row["id"])
        print("Ticket:", row["messages"][1]["content"])
        print("Label:", label)

========== ALL 25 INVALID LABELS ==========

ID: tr-00060
Ticket: Received: 2026-09-17 (Thu)
Source: email

Subject: Complaint

Hi team,
Everytime I try to prove my identity it won't let me. Bank ref no 974846156473.

Regards,
Ananya Bose
Label: {"category": "kyc", "priority": "P3", "amount_paise": null, "txn_date": null, "t

ID: tr-00095
Ticket: Received: 2026-09-15 (Tue)
Source: app-chat

Dear NimbusPay support, May I have another card? My phone is a Realme 11. Thanks.
Label: {"category": "other", "priority": "P3", "amount_paise": null, "txn_date": null, "txn_id": null, "c

ID: tr-00125
Ticket: Received: 2026-08-18 (Tue)
Source: app-chat

Hey, My card payment didn't work. Dated 12-08-26. It is Rs. 3925 in total. The txn id is NP6948115654. My earlier ticket number is #99485. Kindly resolve this asap.
Label: {"category": "payment_failed", "priority": "P3", "amount_paise": 392500, "txn_date": "2026-08-12", "t

ID: tr-00172
Ticket: Received: 2026-09-12 (Sat)
Source: twitter

@NimbusPayC

In [ ]:
import json
import pandas as pd

review_rows = []

for row in train_rows:
    row_id = row["id"]
    ticket = row["messages"][1]["content"]
    raw_label = row["messages"][-1]["content"]

    try:
        label = json.loads(raw_label)

        if set(label.keys()) != EXPECTED_KEYS:
            issue = "Wrong or missing keys"

            review_rows.append({
                "id": row_id,
                "issue": issue,
                "ticket": ticket,
                "raw_label": raw_label
            })

    except json.JSONDecodeError:
        review_rows.append({
            "id": row_id,
            "issue": "Invalid JSON",
            "ticket": ticket,
            "raw_label": raw_label
        })

review_df = pd.DataFrame(review_rows)

print("Total records requiring review:", len(review_df))
print("\nIssue distribution:")
print(review_df["issue"].value_counts())

review_df.to_csv(
    "/content/label_issue_review.csv",
    index=False
)

print("\nReview file saved successfully!")

Total records requiring review: 95

Issue distribution:
issue
Wrong or missing keys    70
Invalid JSON             25
Name: count, dtype: int64

Review file saved successfully!


In [ ]:
from google.colab import files

files.download("/content/label_issue_review.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 4. Label Review and Corrections

Review malformed labels, incorrect keys, transaction IDs, dates, and priority labels. Corrections are made using the training data and schema.

In [ ]:
import json
import pandas as pd

wrong_key_records = []

for row in train_rows:
    label_text = row["messages"][-1]["content"]

    try:
        label = json.loads(label_text)

        if "prio" in label:
            wrong_key_records.append({
                "id": row["id"],
                "ticket": row["messages"][1]["content"],
                "category": label.get("category"),
                "old_priority": label.get("prio"),
                "amount_paise": label.get("amount_paise"),
                "txn_date": label.get("txn_date"),
                "txn_id": label.get("txn_id"),
                "channel": label.get("channel"),
                "language": label.get("language"),
                "needs_human": label.get("needs_human")
            })

    except json.JSONDecodeError:
        pass

wrong_key_df = pd.DataFrame(wrong_key_records)

print("Records:", len(wrong_key_df))
print(wrong_key_df[
    ["id", "category", "old_priority", "amount_paise", "needs_human"]
].to_string(index=False))

Records: 70
      id       category old_priority  amount_paise  needs_human
tr-00020         refund          low       97800.0         True
tr-00049          other          low           NaN        False
tr-00058 payment_failed          low           NaN        False
tr-00072            kyc          low           NaN         True
tr-00077         refund       medium     3700699.0        False
tr-00114          other          low           NaN        False
tr-00120          other          low           NaN        False
tr-00123         refund         high       47700.0         True
tr-00153         refund          low      229000.0        False
tr-00160         refund         high    22810899.0         True
tr-00178          other          low           NaN        False
tr-00192 account_access       medium           NaN        False
tr-00219         offers          low       40700.0        False
tr-00223         offers          low           NaN        False
tr-00249         refund     

In [ ]:
import pandas as pd

def provisional_priority(row):
    category = row["category"]
    amount = row["amount_paise"]

    if category == "fraud":
        return "P1"

    if category == "account_access":
        return "P2"

    if category in ["payment_failed", "refund"]:
        if pd.notna(amount) and amount >= 5000000:
            return "P1"
        if pd.notna(amount) and amount >= 500000:
            return "P2"

    return "P3"


wrong_key_df["provisional_priority"] = wrong_key_df.apply(
    provisional_priority, axis=1
)

print("Total records:", len(wrong_key_df))

for _, row in wrong_key_df.iterrows():
    print("\n" + "=" * 80)
    print("ID:", row["id"])
    print("Category:", row["category"])
    print("Old priority:", row["old_priority"])
    print("Provisional priority:", row["provisional_priority"])
    print("Amount (paise):", row["amount_paise"])
    print("Needs human (old):", row["needs_human"])
    print("TICKET:")
    print(row["ticket"])

Total records: 70

ID: tr-00020
Category: refund
Old priority: low
Provisional priority: P3
Amount (paise): 97800.0
Needs human (old): True
TICKET:
Received: 2026-09-05 (Sat)
Source: ivr-transcript

[IVR transcript] Hey, How can I check if I have received a refund if it is not yet on my statement? 978 rupees is the amount. I have contacted support about this before. Mode: wallet. Using the Android app. This happened on 02-09-26. Thanks.

ID: tr-00049
Category: other
Old priority: low
Provisional priority: P3
Amount (paise): nan
Needs human (old): False
TICKET:
Received: 2026-09-01 (Tue)
Source: ivr-transcript

[IVR transcript]  Regsitered email id kaise change karun? Dhanyavaad.

ID: tr-00058
Category: payment_failed
Old priority: low
Provisional priority: P3
Amount (paise): nan
Needs human (old): False
TICKET:
Received: 2026-09-13 (Sun)
Source: ivr-transcript

[IVR transcript] Dear NimbusPay support, I tried to make a transfer and it didn't work. This happened on Aug 26. Thanks.

ID: 

In [ ]:
priority_review = wrong_key_df.copy()

priority_review["priority_changed"] = (
    priority_review["old_priority"].map({
        "low": "P3",
        "medium": "P2",
        "high": "P1"
    }) != priority_review["provisional_priority"]
)

print("Priority changes suggested:",
      priority_review["priority_changed"].sum())

print("\nSuggested priority distribution:")
print(priority_review["provisional_priority"].value_counts())

print("\nRecords requiring priority review:")

print(
    priority_review.loc[
        priority_review["priority_changed"],
        ["id", "category", "old_priority",
         "provisional_priority", "amount_paise"]
    ].to_string(index=False)
)

Priority changes suggested: 3

Suggested priority distribution:
provisional_priority
P3    46
P2    16
P1     8
Name: count, dtype: int64

Records requiring priority review:
      id       category old_priority provisional_priority  amount_paise
tr-00123         refund         high                   P3       47700.0
tr-00556         refund         high                   P3       23225.0
tr-01151 account_access         high                   P2           NaN


In [ ]:
review_70 = wrong_key_df.copy()

review_70["provisional_priority"] = (
    review_70["provisional_priority"]
)

review_70.to_csv(
    "/content/wrong_key_70_review.csv",
    index=False
)

print("Review file created")
print("Records:", len(review_70))
print(review_70[
    ["id", "ticket", "category", "old_priority",
     "provisional_priority"]
].head(10).to_string(index=False))

Review file created
Records: 70
      id                                                                                                                                                                                                                                                                                                                                                              ticket       category old_priority provisional_priority
tr-00020                                                             Received: 2026-09-05 (Sat)\nSource: ivr-transcript\n\n[IVR transcript] Hey, How can I check if I have received a refund if it is not yet on my statement? 978 rupees is the amount. I have contacted support about this before. Mode: wallet. Using the Android app. This happened on 02-09-26. Thanks.         refund          low                   P3
tr-00049                                                                                                                                    

In [ ]:
import json
import re
import pandas as pd

# Look up the original training records by ID
rows_by_id = {r["id"]: r for r in train_rows}

correction_log = []

legal_pattern = re.compile(
    r"\b(RBI|ombudsman|consumer court|lawyer|legal notice|legal action)\b",
    re.IGNORECASE
)

repeat_pattern = re.compile(
    r"(contacted support before|previous complaint|third time|"
    r"again and again|baar baar|second complaint|earlier ticket)",
    re.IGNORECASE
)

for _, review in wrong_key_df.iterrows():
    rid = review["id"]
    original = rows_by_id[rid]

    ticket = original["messages"][1]["content"]
    assistant_msg = original["messages"][2]
    label = json.loads(assistant_msg["content"])

    old_label = label.copy()

    # Fix the incorrect key
    if "prio" in label:
        old_priority = label.pop("prio")
    else:
        old_priority = label.get("priority")

    # Calculate priority using schema rules
    category = label["category"]
    amount = label.get("amount_paise")

    priority = "P3"

    if category == "fraud" or legal_pattern.search(ticket):
        priority = "P1"

    elif category in ["payment_failed", "refund"]:
        if amount is not None and amount >= 5000000:
            priority = "P1"
        elif amount is not None and amount >= 500000:
            priority = "P2"

    elif category == "account_access":
        priority = "P2"

    label["priority"] = priority

    # P1 always requires human review
    if priority == "P1" or repeat_pattern.search(ticket):
        label["needs_human"] = True

    # Update the original training row
    assistant_msg["content"] = json.dumps(label, ensure_ascii=False)

    correction_log.append({
        "id": rid,
        "action": "corrected",
        "changes": json.dumps({
            "old_priority": old_priority,
            "new_priority": priority,
            "old_keys": list(old_label.keys()),
            "new_keys": list(label.keys())
        })
    })

print("Corrected records:", len(correction_log))
print("Priority distribution:")
print(pd.Series([
    json.loads(rows_by_id[x["id"]]["messages"][2]["content"])["priority"]
    for x in correction_log
]).value_counts())

print("Sample corrected labels:")
for item in correction_log[:5]:
    rid = item["id"]
    print(rid, rows_by_id[rid]["messages"][2]["content"])

Corrected records: 70
Priority distribution:
P3    44
P2    15
P1    11
Name: count, dtype: int64
Sample corrected labels:
tr-00020 {"category": "refund", "amount_paise": 97800, "txn_date": "02/09/2026", "txn_id": null, "channel": "wallet", "language": "en", "needs_human": true, "priority": "P3"}
tr-00049 {"category": "other", "amount_paise": null, "txn_date": null, "txn_id": null, "channel": null, "language": "hi-en", "needs_human": false, "priority": "P3"}
tr-00058 {"category": "payment_failed", "amount_paise": null, "txn_date": "26/08/2026", "txn_id": null, "channel": null, "language": "en", "needs_human": false, "priority": "P3"}
tr-00072 {"category": "kyc", "amount_paise": null, "txn_date": null, "txn_id": null, "channel": null, "language": "hi-en", "needs_human": true, "priority": "P3"}
tr-00077 {"category": "refund", "amount_paise": 3700699, "txn_date": "29/07/2026", "txn_id": null, "channel": "wallet", "language": "hi-en", "needs_human": false, "priority": "P2"}


In [ ]:
from datetime import datetime
import json

date_fixes = 0
date_review = []

for item in correction_log:
    rid = item["id"]
    msg = rows_by_id[rid]["messages"][2]
    label = json.loads(msg["content"])

    date_value = label.get("txn_date")

    if date_value is None:
        continue

    parsed_date = None

    for fmt in [
        "%d/%m/%Y",
        "%d-%m-%Y",
        "%d/%m/%y",
        "%d-%m-%y",
        "%Y-%m-%d"
    ]:
        try:
            parsed_date = datetime.strptime(date_value, fmt)
            break
        except ValueError:
            pass

    if parsed_date:
        new_date = parsed_date.strftime("%Y-%m-%d")

        if new_date != date_value:
            label["txn_date"] = new_date
            msg["content"] = json.dumps(label, ensure_ascii=False)
            date_fixes += 1
    else:
        date_review.append((rid, date_value))

print("Dates normalized:", date_fixes)
print("Dates needing manual review:", date_review)

Dates normalized: 30
Dates needing manual review: []


In [ ]:
import json
import re

id_fixes = 0
invalid_ids = []

for item in correction_log:
    rid = item["id"]
    msg = rows_by_id[rid]["messages"][2]
    label = json.loads(msg["content"])

    txn_id = label.get("txn_id")

    if txn_id is None:
        continue

    # Normalize only recognizable transaction IDs
    cleaned = re.sub(r"[\s-]", "", str(txn_id)).upper()

    if re.fullmatch(r"NP\d{10}", cleaned):
        if cleaned != txn_id:
            label["txn_id"] = cleaned
            msg["content"] = json.dumps(label, ensure_ascii=False)
            id_fixes += 1
    else:
        invalid_ids.append((rid, txn_id))

print("Transaction IDs normalized:", id_fixes)
print("Invalid IDs needing review:", len(invalid_ids))

for item in invalid_ids:
    print(item)

Transaction IDs normalized: 0
Invalid IDs needing review: 0


In [ ]:
import pandas as pd

invalid_ids = [
    "tr-00060", "tr-00095", "tr-00125", "tr-00172",
    "tr-00267", "tr-00357", "tr-00478", "tr-00497",
    "tr-00558", "tr-00605", "tr-00674", "tr-00735",
    "tr-00765", "tr-00809", "tr-00824", "tr-00846",
    "tr-00958", "tr-00960", "tr-00975", "tr-00978",
    "tr-00979", "tr-00980", "tr-01070", "tr-01087",
    "tr-01199"
]

invalid_review = []

for rid in invalid_ids:
    row = rows_by_id[rid]

    invalid_review.append({
        "id": rid,
        "ticket": row["messages"][1]["content"],
        "original_label": row["messages"][2]["content"]
    })

invalid_df = pd.DataFrame(invalid_review)

print(invalid_df.to_string(index=False))

      id                                                                                                                                                                                                                                                                                                                     ticket                                                                                                                            original_label
tr-00060                                                                                                                                      Received: 2026-09-17 (Thu)\nSource: email\n\nSubject: Complaint\n\nHi team,\nEverytime I try to prove my identity it won't let me. Bank ref no 974846156473.\n\nRegards,\nAnanya Bose                                                          {"category": "kyc", "priority": "P3", "amount_paise": null, "txn_date": null, "t
tr-00095                                                                    

In [ ]:
import json
import re
import pandas as pd

repairs = {
    "tr-00060": ["kyc", "P3", None, None, None, None, "en", False],
    "tr-00095": ["other", "P3", None, None, None, None, "en", False],
    "tr-00125": ["payment_failed", "P3", 392500, "2026-08-12", "NP6948115654", "card", "en", True],
    "tr-00172": ["kyc", "P3", None, None, None, None, "en", False],
    "tr-00267": ["kyc", "P3", None, None, None, None, "en", False],
    "tr-00357": ["account_access", "P2", None, None, None, None, "en", False],
    "tr-00478": ["refund", "P1", 16041600, None, "NP1893758181", None, "en", True],
    "tr-00497": ["payment_failed", "P3", 242300, "2026-06-25", None, None, "en", False],
    "tr-00558": ["payment_failed", "P1", 15137000, "2026-07-30", "NP8621659965", "netbanking", "en", True],
    "tr-00605": ["payment_failed", "P3", 37700, "2026-07-25", None, None, "en", False],
    "tr-00674": ["payment_failed", "P3", 338500, "2026-09-16", None, "upi", "en", False],
    "tr-00735": ["payment_failed", "P1", 29687900, None, "NP7764977551", "netbanking", "en", True],
    "tr-00765": ["refund", "P3", 13600, "2026-06-27", "NP4193555354", "card", "hi-en", False],
    "tr-00809": ["payment_failed", "P3", 97000, "2026-07-05", "NP8738912683", "card", "hi-en", False],
    "tr-00824": ["offers", "P3", 27100, None, None, None, "hi-en", False],
    "tr-00846": ["payment_failed", "P3", 23000, None, "NP4657109851", "wallet", "en", False],
    "tr-00958": ["refund", "P3", 59700, None, None, "wallet", "en", True],
    "tr-00960": ["account_access", "P1", None, None, None, None, "hi-en", True],
    "tr-00975": ["fraud", "P1", 94400, "2026-08-29", None, "card", "en", True],
    "tr-00978": ["account_access", "P2", None, None, None, None, "en", True],
    "tr-00979": ["payment_failed", "P3", 83325, "2026-08-07", "NP7335161087", "card", "en", True],
    "tr-00980": ["other", "P3", None, None, None, None, "en", False],
    "tr-01070": ["payment_failed", "P2", 1787400, "2026-08-11", "NP2339235751", "card", "en", False],
    "tr-01087": ["offers", "P3", None, None, None, "card", "en", False],
    "tr-01199": ["payment_failed", "P3", 141099, None, None, "upi", "en", False]
}

KEYS = [
    "category", "priority", "amount_paise", "txn_date",
    "txn_id", "channel", "language", "needs_human"
]

malformed_correction_log = []

for rid, values in repairs.items():
    label = dict(zip(KEYS, values))

    row = rows_by_id[rid]
    row["messages"][2]["content"] = json.dumps(
        label, ensure_ascii=False
    )

    malformed_correction_log.append({
        "id": rid,
        "action": "repaired_malformed_json",
        "reason": "Reconstructed label from ticket using schema"
    })

print("Malformed labels repaired:", len(malformed_correction_log))
print("Sample repaired records:")

for rid in list(repairs.keys())[:5]:
    print(rid, rows_by_id[rid]["messages"][2]["content"])

Malformed labels repaired: 25
Sample repaired records:
tr-00060 {"category": "kyc", "priority": "P3", "amount_paise": null, "txn_date": null, "txn_id": null, "channel": null, "language": "en", "needs_human": false}
tr-00095 {"category": "other", "priority": "P3", "amount_paise": null, "txn_date": null, "txn_id": null, "channel": null, "language": "en", "needs_human": false}
tr-00125 {"category": "payment_failed", "priority": "P3", "amount_paise": 392500, "txn_date": "2026-08-12", "txn_id": "NP6948115654", "channel": "card", "language": "en", "needs_human": true}
tr-00172 {"category": "kyc", "priority": "P3", "amount_paise": null, "txn_date": null, "txn_id": null, "channel": null, "language": "en", "needs_human": false}
tr-00267 {"category": "kyc", "priority": "P3", "amount_paise": null, "txn_date": null, "txn_id": null, "channel": null, "language": "en", "needs_human": false}


## 5. Validation and Clean Dataset Export

Validate the corrected labels, save the cleaned training set, and create the cleaning log.

In [ ]:
EXPECTED_KEYS = set(KEYS)

invalid_json = []
wrong_keys = []
invalid_values = []

for row in train_rows:
    rid = row["id"]

    try:
        label = json.loads(row["messages"][2]["content"])
    except Exception:
        invalid_json.append(rid)
        continue

    if not isinstance(label, dict) or set(label.keys()) != EXPECTED_KEYS:
        wrong_keys.append(rid)
        continue

    if (
        label["category"] not in [
            "payment_failed", "refund", "fraud",
            "account_access", "kyc", "offers", "other"
        ]
        or label["priority"] not in ["P1", "P2", "P3"]
        or label["language"] not in ["en", "hi-en"]
        or type(label["needs_human"]) is not bool
        or (
            label["amount_paise"] is not None
            and type(label["amount_paise"]) is not int
        )
        or (
            label["txn_date"] is not None
            and not re.fullmatch(r"\d{4}-\d{2}-\d{2}", label["txn_date"])
        )
        or (
            label["txn_id"] is not None
            and not re.fullmatch(r"NP\d{10}", label["txn_id"])
        )
        or (
            label["channel"] not in
            ["upi", "card", "netbanking", "wallet", None]
        )
    ):
        invalid_values.append(rid)

print("Total training records:", len(train_rows))
print("Invalid JSON:", len(invalid_json))
print("Wrong keys:", len(wrong_keys))
print("Invalid values/formats:", len(invalid_values))

Total training records: 1445
Invalid JSON: 0
Wrong keys: 0
Invalid values/formats: 20


In [ ]:
import json
import re
import pandas as pd
from datetime import datetime

issue_rows = []

for row in train_rows:
    rid = row["id"]
    label = json.loads(row["messages"][2]["content"])
    ticket = row["messages"][1]["content"]

    issues = []

    if label["category"] not in [
        "payment_failed", "refund", "fraud",
        "account_access", "kyc", "offers", "other"
    ]:
        issues.append("category")

    if label["priority"] not in ["P1", "P2", "P3"]:
        issues.append("priority")

    if label["language"] not in ["en", "hi-en"]:
        issues.append("language")

    if type(label["needs_human"]) is not bool:
        issues.append("needs_human")

    amount = label["amount_paise"]
    if amount is not None and (type(amount) is not int or amount < 0):
        issues.append("amount_paise")

    date = label["txn_date"]
    if date is not None:
        try:
            datetime.strptime(date, "%Y-%m-%d")
        except (ValueError, TypeError):
            issues.append("txn_date")

    txn_id = label["txn_id"]
    if txn_id is not None and (
        not isinstance(txn_id, str)
        or not re.fullmatch(r"NP\d{10}", txn_id)
    ):
        issues.append("txn_id")

    if label["channel"] not in ["upi", "card", "netbanking", "wallet", None]:
        issues.append("channel")

    if issues:
        issue_rows.append({
            "id": rid,
            "issues": ", ".join(issues),
            "ticket": ticket,
            "label": json.dumps(label, ensure_ascii=False)
        })

issues_df = pd.DataFrame(issue_rows)

print("Records requiring correction:", len(issues_df))

for _, r in issues_df.iterrows():
    print("\nID:", r["id"])
    print("Issues:", r["issues"])
    print("Ticket:", r["ticket"])
    print("Label:", r["label"])

Records requiring correction: 20

ID: tr-00071
Issues: txn_id
Ticket: Received: 2026-08-17 (Mon)
Source: email

Subject: Problem with NimbusPay

Hello ji,
Ek hi order ke do baar paise kat gaye, ek wapas kar do. Reference NP-9718716109 hai. Android app use kar raha hoon. 4 din pehle ki baat hai. Rs. 954 kat gaye. Reply ka wait kar raha hoon.

Regards,
Arjun Das
Label: {"category": "refund", "priority": "P3", "amount_paise": 95400, "txn_date": "2026-08-13", "txn_id": "NP-9718716109", "channel": null, "language": "hi-en", "needs_human": false}

ID: tr-00147
Issues: txn_id
Ticket: Received: 2026-07-04 (Sat)
Source: app-chat

Sir, Ek hi order ke do baar paise kat gaye, ek wapas kar do. Jun 22 ki transaction hai. UPI se hua tha. np9891267518 check kar lo. Rs. 859 kat gaye. Mera phone iPhone 13 hai. Jaldi solve karo please.
Label: {"category": "refund", "priority": "P3", "amount_paise": 85900, "txn_date": "2026-06-22", "txn_id": "np9891267518", "channel": "upi", "language": "hi-en", "needs_hu

In [ ]:
import json
import re

txn_id_fixes = []

for row in train_rows:
    label = json.loads(row["messages"][2]["content"])
    old_id = label.get("txn_id")

    if old_id is None:
        continue

    new_id = re.sub(r"[\s-]", "", old_id).upper()

    if new_id != old_id:
        if re.fullmatch(r"NP\d{10}", new_id):
            label["txn_id"] = new_id
            row["messages"][2]["content"] = json.dumps(
                label, ensure_ascii=False
            )

            txn_id_fixes.append({
                "id": row["id"],
                "old_value": old_id,
                "new_value": new_id,
                "action": "normalized_transaction_id"
            })

print("Transaction IDs fixed:", len(txn_id_fixes))
for item in txn_id_fixes:
    print(item)

Transaction IDs fixed: 20
{'id': 'tr-00071', 'old_value': 'NP-9718716109', 'new_value': 'NP9718716109', 'action': 'normalized_transaction_id'}
{'id': 'tr-00147', 'old_value': 'np9891267518', 'new_value': 'NP9891267518', 'action': 'normalized_transaction_id'}
{'id': 'tr-00151', 'old_value': 'NP-2339259875', 'new_value': 'NP2339259875', 'action': 'normalized_transaction_id'}
{'id': 'tr-00391', 'old_value': 'NP 9255627510', 'new_value': 'NP9255627510', 'action': 'normalized_transaction_id'}
{'id': 'tr-00420', 'old_value': 'np 81633 47892', 'new_value': 'NP8163347892', 'action': 'normalized_transaction_id'}
{'id': 'tr-00460', 'old_value': 'NP 6949410784', 'new_value': 'NP6949410784', 'action': 'normalized_transaction_id'}
{'id': 'tr-00470', 'old_value': 'np 11245 46709', 'new_value': 'NP1124546709', 'action': 'normalized_transaction_id'}
{'id': 'tr-00570', 'old_value': 'np 87267 97611', 'new_value': 'NP8726797611', 'action': 'normalized_transaction_id'}
{'id': 'tr-00595', 'old_value': 'np3

In [ ]:
import json
import re
from datetime import datetime

EXPECTED_KEYS = {
    "category", "priority", "amount_paise", "txn_date",
    "txn_id", "channel", "language", "needs_human"
}

allowed = {
    "category": {"payment_failed", "refund", "fraud", "account_access", "kyc", "offers", "other"},
    "priority": {"P1", "P2", "P3"},
    "channel": {"upi", "card", "netbanking", "wallet", None},
    "language": {"en", "hi-en"}
}

issues = []

for row in train_rows:
    try:
        label = json.loads(row["messages"][2]["content"])
    except Exception:
        issues.append((row["id"], "Invalid JSON"))
        continue

    if set(label.keys()) != EXPECTED_KEYS:
        issues.append((row["id"], "Wrong keys"))
        continue

    for field, values in allowed.items():
        if label[field] not in values:
            issues.append((row["id"], f"Invalid {field}: {label[field]}"))

    if label["amount_paise"] is not None and type(label["amount_paise"]) is not int:
        issues.append((row["id"], "Invalid amount_paise"))

    if label["txn_date"] is not None:
        try:
            datetime.strptime(label["txn_date"], "%Y-%m-%d")
        except (ValueError, TypeError):
            issues.append((row["id"], "Invalid txn_date"))

    if label["txn_id"] is not None and not re.fullmatch(r"NP\d{10}", label["txn_id"]):
        issues.append((row["id"], "Invalid txn_id"))

    if type(label["needs_human"]) is not bool:
        issues.append((row["id"], "Invalid needs_human"))

print("Total records:", len(train_rows))
print("Remaining issues:", len(issues))
print(issues[:30])

Total records: 1445
Remaining issues: 0
[]


In [ ]:
import json
import re

semantic_issues = []

legal_pattern = re.compile(
    r"\b(RBI|ombudsman|consumer court|lawyer|legal notice|legal action)\b",
    re.I
)

repeat_pattern = re.compile(
    r"\b(third time|fourth time|again and again|multiple times|"
    r"already contacted|contacted support before|previously contacted)\b",
    re.I
)

fraud_pattern = re.compile(
    r"\b(not me|did not make|didn't make|not made by me|"
    r"unauthorized|unrecognised|unrecognized|without my permission|"
    r"without my approval|don't recognize|do not recognize)\b",
    re.I
)

for row in train_rows:
    label = json.loads(row["messages"][2]["content"])
    ticket = row["messages"][1]["content"]
    rid = row["id"]

    category = label["category"]
    priority = label["priority"]
    amount = label["amount_paise"]

    # Transaction fields must be null for these categories
    if category in ["account_access", "kyc", "other"]:
        for field in ["amount_paise", "txn_date", "txn_id", "channel"]:
            if label[field] is not None:
                semantic_issues.append((rid, f"{field} must be null for {category}"))

    # Priority rules
    if category == "fraud" and priority != "P1":
        semantic_issues.append((rid, "Fraud must be P1"))

    if category == "account_access" and priority == "P3":
        semantic_issues.append((rid, "Account access must be at least P2"))

    if category in ["payment_failed", "refund"] and amount is not None:
        if amount >= 5_000_000 and priority != "P1":
            semantic_issues.append((rid, "Amount >= ₹50,000 must be P1"))
        elif amount >= 500_000 and priority == "P3":
            semantic_issues.append((rid, "Amount >= ₹5,000 must be at least P2"))

    if legal_pattern.search(ticket) and priority != "P1":
        semantic_issues.append((rid, "Legal threat requires P1"))

    if (priority == "P1" or repeat_pattern.search(ticket)) and not label["needs_human"]:
        semantic_issues.append((rid, "needs_human should be true"))

    if fraud_pattern.search(ticket) and category != "fraud":
        semantic_issues.append((rid, "Possible fraud category override — review ticket"))

print("Semantic flags:", len(semantic_issues))
for issue in semantic_issues[:50]:
    print(issue)

Semantic flags: 68
('tr-00018', 'needs_human should be true')
('tr-00023', 'needs_human should be true')
('tr-00045', 'Fraud must be P1')
('tr-00059', 'needs_human should be true')
('tr-00116', 'needs_human should be true')
('tr-00129', 'Amount >= ₹5,000 must be at least P2')
('tr-00134', 'needs_human should be true')
('tr-00146', 'needs_human should be true')
('tr-00154', 'Amount >= ₹5,000 must be at least P2')
('tr-00184', 'Legal threat requires P1')
('tr-00212', 'Amount >= ₹50,000 must be P1')
('tr-00212', 'Legal threat requires P1')
('tr-00220', 'Amount >= ₹5,000 must be at least P2')
('tr-00225', 'needs_human should be true')
('tr-00294', 'Amount >= ₹5,000 must be at least P2')
('tr-00306', 'needs_human should be true')
('tr-00309', 'needs_human should be true')
('tr-00312', 'needs_human should be true')
('tr-00355', 'Account access must be at least P2')
('tr-00365', 'needs_human should be true')
('tr-00380', 'needs_human should be true')
('tr-00413', 'needs_human should be true')

In [ ]:
import json

priority_flags = [
    (rid, reason) for rid, reason in semantic_issues
    if reason != "needs_human should be true"
]

human_flags = [
    rid for rid, reason in semantic_issues
    if reason == "needs_human should be true"
]

print("=== PRIORITY / CATEGORY FLAGS ===")
for rid, reason in priority_flags:
    row = rows_by_id[rid]
    label = json.loads(row["messages"][2]["content"])
    ticket = row["messages"][1]["content"]

    print(f"\n{rid} | {reason}")
    print("LABEL:", label)
    print("TICKET:", ticket[:700])

print("\n=== FIRST 10 NEEDS_HUMAN FLAGS ===")
for rid in human_flags[:10]:
    row = rows_by_id[rid]
    label = json.loads(row["messages"][2]["content"])

    print(f"\n{rid} | needs_human={label['needs_human']}")
    print("TICKET:", row["messages"][1]["content"][:500])

=== PRIORITY / CATEGORY FLAGS ===

tr-00045 | Fraud must be P1
LABEL: {'category': 'fraud', 'priority': 'P3', 'amount_paise': 1159400, 'txn_date': None, 'txn_id': None, 'channel': None, 'language': 'en', 'needs_human': True}
TICKET: Received: 2026-06-01 (Mon)
Source: email

Subject: Help needed


I AM MISSING CASH FROM MY BANK ACCOUNT ACCORDING TO MY APP. The amount was INR 11,594. I am on the iOS app. Waiting for your reply.

Regards,
Fatima Sheikh

tr-00129 | Amount >= ₹5,000 must be at least P2
LABEL: {'category': 'payment_failed', 'priority': 'P3', 'amount_paise': 4619100, 'txn_date': '2026-09-01', 'txn_id': None, 'channel': 'card', 'language': 'hi-en', 'needs_human': False}
TICKET: Received: 2026-09-07 (Mon)
Source: email

Subject: Issue with my account

Sir,
Recharge nahi hua lekin paise kat gaye hain. Mera phone Samsung M34 hai. Amount ₹46,191 tha. Yeh 1 Sep ko hua. Card se hua tha. Dhanyavaad.

Regards,
Pooja Nair

tr-00154 | Amount >= ₹5,000 must be at least P2
LABEL: {'catego

In [ ]:
import json
import re

legal_pattern = re.compile(
    r"\b(RBI|ombudsman|consumer court|lawyer|legal notice|legal action)\b",
    re.I
)

priority_fixes = []

for row in train_rows:
    label = json.loads(row["messages"][2]["content"])
    ticket = row["messages"][1]["content"]

    old_priority = label["priority"]
    old_human = label["needs_human"]

    category = label["category"]
    amount = label["amount_paise"]

    # Apply priority rules
    if category == "fraud" or legal_pattern.search(ticket):
        new_priority = "P1"
    elif category == "account_access":
        new_priority = "P2"
    elif category in ["payment_failed", "refund"]:
        if amount is not None and amount >= 5_000_000:
            new_priority = "P1"
        elif amount is not None and amount >= 500_000:
            new_priority = "P2"
        else:
            new_priority = "P3"
    else:
        new_priority = "P3"

    label["priority"] = new_priority

    if new_priority == "P1":
        label["needs_human"] = True

    if old_priority != label["priority"] or old_human != label["needs_human"]:
        priority_fixes.append({
            "id": row["id"],
            "old_priority": old_priority,
            "new_priority": label["priority"],
            "old_needs_human": old_human,
            "new_needs_human": label["needs_human"]
        })

    row["messages"][2]["content"] = json.dumps(
        label, ensure_ascii=False
    )

print("Priority records corrected:", len(priority_fixes))
for fix in priority_fixes:
    print(fix)

print("\nUpdated priority distribution:")
print(pd.Series([
    json.loads(row["messages"][2]["content"])["priority"]
    for row in train_rows
]).value_counts())

Priority records corrected: 65
{'id': 'tr-00013', 'old_priority': 'P1', 'new_priority': 'P3', 'old_needs_human': True, 'new_needs_human': True}
{'id': 'tr-00018', 'old_priority': 'P1', 'new_priority': 'P3', 'old_needs_human': False, 'new_needs_human': False}
{'id': 'tr-00023', 'old_priority': 'P1', 'new_priority': 'P3', 'old_needs_human': False, 'new_needs_human': False}
{'id': 'tr-00045', 'old_priority': 'P3', 'new_priority': 'P1', 'old_needs_human': True, 'new_needs_human': True}
{'id': 'tr-00059', 'old_priority': 'P1', 'new_priority': 'P3', 'old_needs_human': False, 'new_needs_human': False}
{'id': 'tr-00073', 'old_priority': 'P1', 'new_priority': 'P3', 'old_needs_human': True, 'new_needs_human': True}
{'id': 'tr-00129', 'old_priority': 'P3', 'new_priority': 'P2', 'old_needs_human': False, 'new_needs_human': False}
{'id': 'tr-00146', 'old_priority': 'P1', 'new_priority': 'P3', 'old_needs_human': False, 'new_needs_human': False}
{'id': 'tr-00154', 'old_priority': 'P3', 'new_priority'

In [ ]:
import json

demoted_ids = [
    f["id"] for f in priority_fixes
    if f["old_priority"] == "P1" and f["new_priority"] != "P1"
]

print("P1 records demoted:", len(demoted_ids))

for rid in demoted_ids:
    row = rows_by_id[rid]
    label = json.loads(row["messages"][2]["content"])

    print("\n", rid)
    print("Current label:", label)
    print("Ticket:", row["messages"][1]["content"][:650])

P1 records demoted: 33

 tr-00013
Current label: {'category': 'kyc', 'priority': 'P3', 'amount_paise': None, 'txn_date': None, 'txn_id': None, 'channel': None, 'language': 'hi-en', 'needs_human': True}
Ticket: Received: 2026-08-22 (Sat)
Source: email

Subject: Issue with my account

Hi,
KYC mein address update karna hai. Android app use kar raha hoon. Bank ref 782509120860 hai. Pehle bhi complaint kar chuka hoon.

Regards,
Meera Joshi

 tr-00018
Current label: {'category': 'offers', 'priority': 'P3', 'amount_paise': 25000, 'txn_date': None, 'txn_id': None, 'channel': 'upi', 'language': 'en', 'needs_human': False}
Ticket: Received: 2026-08-01 (Sat)
Source: email

Subject: Issue with my account

Dear NimbusPay support,
Referral bonus was not received after my friend signed up. Mode: UPI. My phone is a OnePlus Nord. The transaction amount is Rs. 250. Thanks.

Regards,
Imran Khan

 tr-00023
Current label: {'category': 'kyc', 'priority': 'P3', 'amount_paise': None, 'txn_date': None, 'txn_id

In [ ]:
import json

amount_fixes = {
    "tr-00073": 16878725,
    "tr-00187": 22000000
}

for row in train_rows:
    if row["id"] in amount_fixes:
        label = json.loads(row["messages"][2]["content"])
        label["amount_paise"] = amount_fixes[row["id"]]
        label["priority"] = "P1"
        label["needs_human"] = True

        row["messages"][2]["content"] = json.dumps(
            label, ensure_ascii=False
        )

        print(row["id"], label)

tr-00073 {'category': 'payment_failed', 'priority': 'P1', 'amount_paise': 16878725, 'txn_date': None, 'txn_id': 'NP4271342441', 'channel': 'card', 'language': 'en', 'needs_human': True}
tr-00187 {'category': 'payment_failed', 'priority': 'P1', 'amount_paise': 22000000, 'txn_date': '2026-08-03', 'txn_id': 'NP1220851898', 'channel': 'upi', 'language': 'hi-en', 'needs_human': True}


In [ ]:
import json

amount_fixes = {
    "tr-00073": 16878725,
    "tr-00187": 22000000
}

amount_correction_log = []

for row in train_rows:
    rid = row["id"]

    if rid in amount_fixes:
        label = json.loads(row["messages"][2]["content"])

        old_amount = label["amount_paise"]
        old_priority = label["priority"]

        label["amount_paise"] = amount_fixes[rid]
        label["priority"] = "P1"
        label["needs_human"] = True

        row["messages"][2]["content"] = json.dumps(
            label, ensure_ascii=False
        )

        amount_correction_log.append({
            "id": rid,
            "old_amount_paise": old_amount,
            "new_amount_paise": amount_fixes[rid],
            "old_priority": old_priority,
            "new_priority": "P1",
            "action": "corrected_amount_and_priority"
        })

print("Amount corrections:", len(amount_correction_log))
print(amount_correction_log)

Amount corrections: 2
[{'id': 'tr-00073', 'old_amount_paise': 16878725, 'new_amount_paise': 16878725, 'old_priority': 'P1', 'new_priority': 'P1', 'action': 'corrected_amount_and_priority'}, {'id': 'tr-00187', 'old_amount_paise': 22000000, 'new_amount_paise': 22000000, 'old_priority': 'P1', 'new_priority': 'P1', 'action': 'corrected_amount_and_priority'}]


In [ ]:
import json
import re

issues = []

legal_pattern = re.compile(
    r"\b(RBI|ombudsman|consumer court|lawyer|legal notice|legal action)\b",
    re.I
)

for row in train_rows:
    label = json.loads(row["messages"][2]["content"])
    ticket = row["messages"][1]["content"]
    rid = row["id"]

    category = label["category"]
    priority = label["priority"]
    amount = label["amount_paise"]

    expected = None

    if category == "fraud" or legal_pattern.search(ticket):
        expected = "P1"
    elif category == "account_access":
        expected = "P2"
    elif category in ["payment_failed", "refund"]:
        if amount is not None and amount >= 5_000_000:
            expected = "P1"
        elif amount is not None and amount >= 500_000:
            expected = "P2"
        else:
            expected = "P3"
    else:
        expected = "P3"

    if priority != expected:
        issues.append({
            "id": rid,
            "issue": "priority",
            "current": priority,
            "expected": expected,
            "ticket": ticket[:250]
        })

    if priority == "P1" and not label["needs_human"]:
        issues.append({
            "id": rid,
            "issue": "needs_human should be true",
            "ticket": ticket[:250]
        })

    if category in ["account_access", "kyc", "other"]:
        fields = ["amount_paise", "txn_date", "txn_id", "channel"]
        non_null = [f for f in fields if label[f] is not None]

        if non_null:
            issues.append({
                "id": rid,
                "issue": "transaction fields should be null",
                "fields": non_null
            })

print("Remaining semantic issues:", len(issues))
for item in issues[:20]:
    print(item)

Remaining semantic issues: 0


In [ ]:
import json
import pandas as pd
import os

# 1. Save cleaned training data
with open("/content/train_clean.jsonl", "w", encoding="utf-8") as f:
    for row in train_rows:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

# 2. Combine correction logs
log_sources = [
    ("malformed_correction_log", "malformed_json_repair"),
    ("correction_log", "wrong_key_correction"),
    ("txn_id_fixes", "transaction_id_normalization"),
    ("priority_fixes", "priority_correction"),
    ("date_fixes", "date_normalization"),
]

log_entries = []

for variable, action_name in log_sources:
    entries = globals().get(variable, [])
    if isinstance(entries, list):
        for entry in entries:
            if isinstance(entry, dict):
                log_entries.append({
                    "id": entry.get("id"),
                    "action": entry.get("action", action_name),
                    "details": json.dumps(entry, ensure_ascii=False, default=str)
                })

# Include amount log only when a value actually changed
for entry in globals().get("amount_correction_log", []):
    if entry.get("old_amount_paise") != entry.get("new_amount_paise"):
        log_entries.append({
            "id": entry.get("id"),
            "action": "amount_correction",
            "details": json.dumps(entry, ensure_ascii=False, default=str)
        })

cleaning_log = pd.DataFrame(
    log_entries,
    columns=["id", "action", "details"]
)

cleaning_log.to_csv("/content/cleaning_log.csv", index=False)

print("Cleaned training records:", len(train_rows))
print("Cleaning log entries:", len(cleaning_log))
print("Cleaned dataset size:", os.path.getsize("/content/train_clean.jsonl"))
print("Cleaning log size:", os.path.getsize("/content/cleaning_log.csv"))
print("\nLog actions:")
print(cleaning_log["action"].value_counts())

Cleaned training records: 1445
Cleaning log entries: 180
Cleaned dataset size: 1165245
Cleaning log size: 49280

Log actions:
action
corrected                    70
priority_correction          65
repaired_malformed_json      25
normalized_transaction_id    20
Name: count, dtype: int64


In [ ]:
print("date_fixes exists:", "date_fixes" in globals())
if "date_fixes" in globals():
    print("Date fix entries:", len(date_fixes))
    print(date_fixes[:3])

date_fixes exists: True


TypeError: object of type 'int' has no len()

In [ ]:
import json
import pandas as pd

date_log = []

for _, old_row in wrong_key_df.iterrows():
    rid = old_row["id"]

    current_row = next(
        (r for r in train_rows if r["id"] == rid),
        None
    )

    if current_row is None:
        continue

    current_label = json.loads(current_row["messages"][2]["content"])

    old_date = old_row["txn_date"]
    new_date = current_label["txn_date"]

    old_date = None if pd.isna(old_date) else str(old_date)
    new_date = None if pd.isna(new_date) else str(new_date)

    if old_date != new_date:
        date_log.append({
            "id": rid,
            "action": "date_normalization",
            "details": json.dumps({
                "old_txn_date": old_date,
                "new_txn_date": new_date
            })
        })

print("Reconstructed date corrections:", len(date_log))
print(date_log[:5])

Reconstructed date corrections: 30
[{'id': 'tr-00020', 'action': 'date_normalization', 'details': '{"old_txn_date": "02/09/2026", "new_txn_date": "2026-09-02"}'}, {'id': 'tr-00058', 'action': 'date_normalization', 'details': '{"old_txn_date": "26/08/2026", "new_txn_date": "2026-08-26"}'}, {'id': 'tr-00077', 'action': 'date_normalization', 'details': '{"old_txn_date": "29/07/2026", "new_txn_date": "2026-07-29"}'}, {'id': 'tr-00123', 'action': 'date_normalization', 'details': '{"old_txn_date": "21/07/2026", "new_txn_date": "2026-07-21"}'}, {'id': 'tr-00153', 'action': 'date_normalization', 'details': '{"old_txn_date": "04/09/2026", "new_txn_date": "2026-09-04"}'}]


In [ ]:
date_log_df = pd.DataFrame(date_log)

cleaning_log = pd.concat(
    [cleaning_log, date_log_df],
    ignore_index=True
)

cleaning_log.to_csv("/content/cleaning_log.csv", index=False)

print("Updated cleaning log:", len(cleaning_log))
print(cleaning_log["action"].value_counts())

Updated cleaning log: 210
action
corrected                    70
priority_correction          65
date_normalization           30
repaired_malformed_json      25
normalized_transaction_id    20
Name: count, dtype: int64


In [ ]:
from google.colab import files

files.download("/content/cleaning_log.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
files.download("/content/train_clean.jsonl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 6. Runtime, GPU, and File Setup

Check the Colab runtime, GPU, required files, and installed libraries. Some cells in this section are exploratory upload or environment checks.

In [ ]:
import os
import json
import torch

print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("\nRequired files:")
for path in [
    "/content/train_clean.jsonl",
    "/content/dev.jsonl",
    "/content/test_inputs.jsonl",
    "/content/score.py"
]:
    print(path, "->", os.path.exists(path))

# Load clean dev data
with open("/content/dev.jsonl", "r", encoding="utf-8") as f:
    dev_rows = [json.loads(line) for line in f if line.strip()]

print("\nDev records:", len(dev_rows))
print("\nSample ID:", dev_rows[0]["id"])
print("\nSystem prompt:")
print(dev_rows[0]["messages"][0]["content"])

GPU available: False

Required files:
/content/train_clean.jsonl -> True
/content/dev.jsonl -> True
/content/test_inputs.jsonl -> False
/content/score.py -> False

Dev records: 200

Sample ID: dv-00001

System prompt:
You are NimbusPay's ticket triage model. Read the ticket and return the triage JSON.


In [ ]:
import os
import torch
import bitsandbytes as bnb
import transformers
import transformers.utils.import_utils as iu

print("GPU available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))
print("BitsAndBytes:", bnb.__version__)
print("Transformers:", transformers.__version__)
print("BitsAndBytes detected:", iu.is_bitsandbytes_available())

print("\nFiles available:")
print(os.listdir("/content"))

GPU available: True
GPU: Tesla T4
BitsAndBytes: 0.50.2
Transformers: 5.22.1
BitsAndBytes detected: True

Files available:
['.config', 'score.py', 'dev.jsonl', 'train.jsonl', 'test_inputs.jsonl', 'train_clean.jsonl', 'sample_data']


In [ ]:
from google.colab import files

uploaded = files.upload()

Saving dev.jsonl to dev.jsonl


In [ ]:
import torch

print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))
    !nvidia-smi

GPU available: True
GPU name: Tesla T4
Thu Nov 12 06:48:54 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   34C    P8             12W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+--------

In [ ]:
from google.colab import files

uploaded = files.upload()

Saving test_inputs.jsonl to test_inputs.jsonl


In [ ]:
import os

for file in [
    "train_clean.jsonl",
    "dev.jsonl",
    "test_inputs.jsonl",
    "score.py"
]:
    print(file, "->", os.path.exists("/content/" + file))

train_clean.jsonl -> True
dev.jsonl -> True
test_inputs.jsonl -> True
score.py -> True


In [ ]:
from google.colab import files

uploaded = files.upload()

Saving train_clean.jsonl to train_clean.jsonl


In [ ]:
!pip install -q transformers accelerate peft bitsandbytes datasets sentencepiece

In [ ]:
with open("/content/score.py", "r", encoding="utf-8") as f:
    print(f.read())

"""Score triage predictions against labelled data.

    python score.py --gold data/dev.jsonl --pred my_dev_predictions.jsonl

Prediction file: one JSON object per line
    {"id": "dv-00001", "output": "<the raw text your model generated>"}

Scoring is strict:
  * "output" must be ONLY a JSON object (leading/trailing whitespace is ignored).
  * Every field must match exactly, including type (true is not 1, 129900 is not "129900").
  * A missing id or invalid JSON counts as wrong on every field.
"""
import argparse, json
from collections import defaultdict

KEYS = ["category", "priority", "amount_paise", "txn_date", "txn_id", "channel", "language", "needs_human"]


def load(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]


def same(a, b):
    return type(a) is type(b) and a == b


def parse(text):
    try:
        obj = json.loads(text.strip())
    except (json.JSONDecodeError, AttributeError):
        return None
    re

In [ ]:
import transformers
import peft
import bitsandbytes
import datasets
import accelerate

print("Transformers:", transformers.__version__)
print("PEFT:", peft.__version__)
print("BitsAndBytes:", bitsandbytes.__version__)
print("Datasets:", datasets.__version__)
print("Accelerate:", accelerate.__version__)

Transformers: 5.22.1
PEFT: 0.21.2
BitsAndBytes: 0.50.2
Datasets: 4.8.5
Accelerate: 1.15.0


## 7. Baseline Evaluations

Evaluate the untuned base model with the fixed prompt, then with schema rules added to the prompt as a reference baseline.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Load model in FP16
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto"
)

model.eval()

print("Model loaded:", MODEL_NAME)
print("Model device:", model.device)
print("GPU memory allocated:",
      round(torch.cuda.memory_allocated() / 1024**3, 2), "GB")

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded: Qwen/Qwen2.5-1.5B-Instruct
Model device: cuda:0
GPU memory allocated: 2.88 GB


In [ ]:
import json
import torch
from tqdm.auto import tqdm

baseline1_predictions = []

model.eval()

for row in tqdm(dev_rows, desc="Baseline 1"):
    messages = [
        {"role": msg["role"], "content": msg["content"]}
        for msg in row["messages"]
        if msg["role"] in ["system", "user"]
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            inputs,
            max_new_tokens=200,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    generated_tokens = outputs[0][inputs.shape[-1]:]
    generated_text = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

    baseline1_predictions.append({
        "id": row["id"],
        "output": generated_text
    })

with open("/content/baseline1_predictions.jsonl", "w", encoding="utf-8") as f:
    for pred in baseline1_predictions:
        f.write(json.dumps(pred, ensure_ascii=False) + "\n")

print("Baseline 1 predictions:", len(baseline1_predictions))
print("Sample prediction:")
print(baseline1_predictions[0]["output"])

NameError: name 'dev_rows' is not defined

In [ ]:
import json

with open("/content/dev.jsonl", "r", encoding="utf-8") as f:
    dev_rows = [json.loads(line) for line in f if line.strip()]

print("Dev records loaded:", len(dev_rows))
print("First ID:", dev_rows[0]["id"])

Dev records loaded: 200
First ID: dv-00001


In [ ]:
import json
import torch
from tqdm.auto import tqdm

baseline1_predictions = []

model.eval()

for row in tqdm(dev_rows, desc="Baseline 1"):
    messages = [
        {"role": msg["role"], "content": msg["content"]}
        for msg in row["messages"]
        if msg["role"] in ["system", "user"]
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True
    )

    inputs = inputs.to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            max_new_tokens=200,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    generated_tokens = outputs[0][inputs["input_ids"].shape[-1]:]

    generated_text = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

    baseline1_predictions.append({
        "id": row["id"],
        "output": generated_text
    })

with open("/content/baseline1_predictions.jsonl", "w", encoding="utf-8") as f:
    for pred in baseline1_predictions:
        f.write(json.dumps(pred, ensure_ascii=False) + "\n")

print("Baseline 1 predictions:", len(baseline1_predictions))
print("\nSample prediction:")
print(baseline1_predictions[0]["output"])

Baseline 1:   0%|          | 0/200 [00:00<?, ?it/s]

Baseline 1 predictions: 200

Sample prediction:
```json
{
  "status": "New",
  "priority": "High",
  "category": "Card Activation",
  "description": "User received an activation request for their card but has not yet been able to use it. They need assistance with activating and using the card.",
  "ticket_number": "#54047"
}
```


In [ ]:
!python /content/score.py --gold /content/dev.jsonl --pred /content/baseline1_predictions.jsonl

group                            n   valid   exact  fields
all                            200    0.0%    0.0%    0.0%
category=account_access         19    0.0%    0.0%    0.0%
category=fraud                  15    0.0%    0.0%    0.0%
category=kyc                    24    0.0%    0.0%    0.0%
category=offers                 18    0.0%    0.0%    0.0%
category=other                  16    0.0%    0.0%    0.0%
category=payment_failed         53    0.0%    0.0%    0.0%
category=refund                 55    0.0%    0.0%    0.0%

per-field accuracy (all rows)
  category         0.0%
  priority         0.0%
  amount_paise     0.0%
  txn_date         0.0%
  txn_id           0.0%
  channel          0.0%
  language         0.0%
  needs_human      0.0%


In [ ]:
import json
import torch
from tqdm.auto import tqdm

baseline2_predictions = []
model.eval()

# Preserve the original system prompt and add NimbusPay rules
original_system_prompt = dev_rows[0]["messages"][0]["content"]

schema_rules = """
You must classify each NimbusPay support ticket and return exactly one JSON object.

The JSON must contain exactly these 8 keys:
category, priority, amount_paise, txn_date, txn_id, channel, language, needs_human.

CATEGORY:
Choose one: payment_failed, refund, fraud, account_access, kyc, offers, other.
Fraud overrides all other categories when the customer reports an unauthorized transaction.

PRIORITY:
P1: fraud, disputed amount >= 50000 rupees, or legal threats involving RBI,
ombudsman, consumer court, lawyer, legal notice, or legal action.
P2: payment_failed or refund with amount >= 5000 rupees, or account_access.
P3: otherwise.

FIELDS:
amount_paise: transaction amount in paise (1 rupee = 100 paise), or null.
txn_date: transaction date in YYYY-MM-DD format, or null.
txn_id: NP followed by exactly 10 digits, or null.
channel: upi, card, netbanking, wallet, or null.
language: hi-en for Hindi written in Latin script or mixed Hindi-English;
otherwise en.
needs_human: true if P1 or the customer has contacted support previously
about the same issue; otherwise false.

For account_access, kyc, and other, amount_paise, txn_date,
txn_id, and channel must all be null.

Resolve relative dates using the ticket's Received date.
Ignore quoted earlier emails when identifying the transaction date.
Use null when information is not provided.

Return only valid JSON. Do not include Markdown fences,
explanations, or additional keys.
"""

for row in tqdm(dev_rows, desc="Baseline 2"):
    messages = [
        {
            "role": "system",
            "content": original_system_prompt + "\n\n" + schema_rules
        }
    ]

    for msg in row["messages"]:
        if msg["role"] == "user":
            messages.append({
                "role": "user",
                "content": msg["content"]
            })

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True
    )

    inputs = inputs.to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            max_new_tokens=200,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    generated_tokens = outputs[0][inputs["input_ids"].shape[-1]:]

    generated_text = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

    baseline2_predictions.append({
        "id": row["id"],
        "output": generated_text
    })

with open("/content/baseline2_predictions.jsonl", "w", encoding="utf-8") as f:
    for pred in baseline2_predictions:
        f.write(json.dumps(pred, ensure_ascii=False) + "\n")

print("Baseline 2 predictions:", len(baseline2_predictions))
print("\nSample prediction:")
print(baseline2_predictions[0]["output"])

Baseline 2:   0%|          | 0/200 [00:00<?, ?it/s]

Baseline 2 predictions: 200

Sample prediction:
```json
{
  "category": "account_access",
  "priority": "P3",
  "amount_paise": null,
  "txn_date": null,
  "txn_id": null,
  "channel": null,
  "language": null,
  "needs_human": true
}
```


In [ ]:
!python /content/score.py --gold /content/dev.jsonl --pred /content/baseline2_predictions.jsonl

group                            n   valid   exact  fields
all                            200    0.0%    0.0%    0.0%
category=account_access         19    0.0%    0.0%    0.0%
category=fraud                  15    0.0%    0.0%    0.0%
category=kyc                    24    0.0%    0.0%    0.0%
category=offers                 18    0.0%    0.0%    0.0%
category=other                  16    0.0%    0.0%    0.0%
category=payment_failed         53    0.0%    0.0%    0.0%
category=refund                 55    0.0%    0.0%    0.0%

per-field accuracy (all rows)
  category         0.0%
  priority         0.0%
  amount_paise     0.0%
  txn_date         0.0%
  txn_id           0.0%
  channel          0.0%
  language         0.0%
  needs_human      0.0%


In [ ]:
print(repr(baseline2_predictions[0]["output"]))
print("\nLength:", len(baseline2_predictions[0]["output"]))
print("\nFirst 5 raw predictions:")
for p in baseline2_predictions[:5]:
    print(p["id"], repr(p["output"][:150]))

'```json\n{\n  "category": "account_access",\n  "priority": "P3",\n  "amount_paise": null,\n  "txn_date": null,\n  "txn_id": null,\n  "channel": null,\n  "language": null,\n  "needs_human": true\n}\n```'

Length: 190

First 5 raw predictions:
dv-00001 '```json\n{\n  "category": "account_access",\n  "priority": "P3",\n  "amount_paise": null,\n  "txn_date": null,\n  "txn_id": null,\n  "channel": null,\n  "lang'
dv-00002 '```json\n{\n  "category": "other",\n  "priority": "P3",\n  "amount_paise": null,\n  "txn_date": null,\n  "txn_id": null,\n  "channel": null,\n  "language": nu'
dv-00003 '```json\n{\n  "category": "payment_failed",\n  "priority": "P1",\n  "amount_paise": 228300,\n  "txn_date": "2026-09-18",\n  "txn_id": "NP1234567890",\n  "cha'
dv-00004 '```json\n{\n  "category": "account_access",\n  "priority": "P3",\n  "amount_paise": null,\n  "txn_date": null,\n  "txn_id": null,\n  "channel": null,\n  "lang'
dv-00005 '```json\n{\n  "category": "payment_failed",\n  "priority": "P1",\n  

## 8. Sequence-Length Analysis

Measure tokenized conversation lengths and verify that the prompt tokens are an exact prefix of the full training conversation.

In [ ]:
import json
import numpy as np

with open("/content/train_clean.jsonl", "r", encoding="utf-8") as f:
    train_rows = [json.loads(line) for line in f if line.strip()]

full_lengths = []
prefix_mismatches = 0

for row in train_rows:
    messages = row["messages"]

    full_ids = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=False
    )

    prompt_ids = tokenizer.apply_chat_template(
        messages[:-1],
        tokenize=True,
        add_generation_prompt=True
    )

    full_lengths.append(len(full_ids))

    if full_ids[:len(prompt_ids)] != prompt_ids:
        prefix_mismatches += 1

print("Training records:", len(train_rows))
print("Sequence length percentiles:")
for p in [50, 90, 95, 99, 100]:
    print(f"{p}th percentile:", int(np.percentile(full_lengths, p)))

print("Maximum length:", max(full_lengths))
print("Assistant masking prefix mismatches:", prefix_mismatches)

NameError: name 'tokenizer' is not defined

In [ ]:
import os

print("Files in /content:")
print(os.listdir("/content"))

Files in /content:
['.config', 'sample_data']


In [ ]:
from google.colab import files

uploaded = files.upload()

Saving dev.jsonl to dev.jsonl


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

print("GPU available:", torch.cuda.is_available())

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto"
)

model.eval()

print("Model loaded:", MODEL_NAME)
print("Device:", model.device)

GPU available: True


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

... (2 outputs left out)

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded: Qwen/Qwen2.5-1.5B-Instruct
Device: cuda:0


In [ ]:
import numpy as np

full_lengths = []
prefix_mismatches = 0

for row in train_rows:
    messages = row["messages"]

    full_ids = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=False
    )

    prompt_ids = tokenizer.apply_chat_template(
        messages[:-1],
        tokenize=True,
        add_generation_prompt=True
    )

    full_lengths.append(len(full_ids))

    if full_ids[:len(prompt_ids)] != prompt_ids:
        prefix_mismatches += 1

print("Training records:", len(train_rows))
print("Sequence length percentiles:")

for p in [50, 90, 95, 99, 100]:
    print(f"{p}th percentile:", int(np.percentile(full_lengths, p)))

print("Maximum length:", max(full_lengths))
print("Prefix mismatches:", prefix_mismatches)

Training records: 1445
Sequence length percentiles:
50th percentile: 2
90th percentile: 2
95th percentile: 2
99th percentile: 2
100th percentile: 2
Maximum length: 2
Prefix mismatches: 1445


In [ ]:
sample = train_rows[0]

print("Message roles:")
for msg in sample["messages"]:
    print(msg["role"], len(msg["content"]))

result = tokenizer.apply_chat_template(
    sample["messages"],
    tokenize=True,
    add_generation_prompt=False
)

print("\nReturn type:", type(result))
print("Return value:", result)

Message roles:
system 84
user 277
assistant 189

Return type: <class 'transformers.tokenization_utils_base.BatchEncoding'>
Return value: {'input_ids': [151644, 8948, 198, 2610, 525, 71493, 20595, 594, 11727, 2406, 424, 1614, 13, 4457, 279, 11727, 323, 470, 279, 2406, 424, 4718, 13, 151645, 198, 151644, 872, 198, 23260, 25, 220, 17, 15, 17, 21, 12, 15, 22, 12, 17, 20, 320, 34882, 340, 3608, 25, 906, 60535, 271, 13048, 11, 358, 614, 264, 1909, 705, 429, 3207, 944, 728, 1526, 13, 8429, 30, 1084, 572, 3351, 13, 14562, 25, 7602, 22798, 13, 3017, 4540, 374, 264, 3731, 8155, 7036, 220, 16, 17, 13, 1084, 374, 220, 19, 13, 22, 74, 304, 2790, 13, 3017, 2692, 8172, 374, 220, 17, 20, 23, 11122, 81300, 1290, 1431, 13, 576, 49721, 877, 374, 43240, 16, 20, 24, 24, 15, 19, 21, 18, 18, 17, 13, 52186, 369, 697, 9851, 13, 151645, 198, 151644, 77091, 198, 4913, 5471, 788, 330, 13755, 35060, 497, 330, 23582, 788, 330, 47, 18, 497, 330, 6045, 620, 4016, 788, 220, 19, 22, 15, 15, 15, 15, 11, 330, 73370, 4164

In [ ]:
import numpy as np

full_lengths = []
prefix_mismatches = 0

for row in train_rows:
    messages = row["messages"]

    full_enc = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=False,
        return_dict=True
    )

    prompt_enc = tokenizer.apply_chat_template(
        messages[:-1],
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True
    )

    full_ids = full_enc["input_ids"]
    prompt_ids = prompt_enc["input_ids"]

    full_lengths.append(len(full_ids))

    if full_ids[:len(prompt_ids)] != prompt_ids:
        prefix_mismatches += 1

print("Training records:", len(train_rows))

for p in [50, 90, 95, 99, 100]:
    print(f"{p}th percentile:", int(np.percentile(full_lengths, p)))

print("Maximum length:", max(full_lengths))
print("Prefix mismatches:", prefix_mismatches)

Training records: 1445
50th percentile: 168
90th percentile: 214
95th percentile: 225
99th percentile: 1804
100th percentile: 2215
Maximum length: 2215
Prefix mismatches: 0


In [ ]:
import pandas as pd

length_data = []

for row in train_rows:
    messages = row["messages"]

    full_ids = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=False
    )["input_ids"]

    prompt_ids = tokenizer.apply_chat_template(
        messages[:-1],
        tokenize=True,
        add_generation_prompt=True
    )["input_ids"]

    length_data.append({
        "id": row["id"],
        "total_tokens": len(full_ids),
        "prompt_tokens": len(prompt_ids),
        "answer_tokens": len(full_ids) - len(prompt_ids)
    })

length_df = pd.DataFrame(length_data)

for limit in [256, 512, 1024, 2048]:
    count = (length_df["total_tokens"] <= limit).sum()
    print(f"{limit} tokens: {count}/1445 examples fit ({count/1445*100:.2f}%)")

print("\nLongest 10 examples:")
print(length_df.nlargest(10, "total_tokens").to_string(index=False))

256 tokens: 1401/1445 examples fit (96.96%)
512 tokens: 1405/1445 examples fit (97.23%)
1024 tokens: 1405/1445 examples fit (97.23%)
2048 tokens: 1436/1445 examples fit (99.38%)

Longest 10 examples:
      id  total_tokens  prompt_tokens  answer_tokens
tr-00817          2215           2145             70
tr-00459          2211           2129             82
tr-01095          2174           2109             65
tr-00576          2169           2104             65
tr-01233          2130           2071             59
tr-00524          2116           2064             52
tr-00654          2070           1989             81
tr-00566          2062           1981             81
tr-01333          2061           1978             83
tr-00452          2043           1979             64


In [ ]:
MAX_LENGTH = 2304

train_examples = []

for row in train_rows:
    messages = row["messages"]

    full_ids = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=False
    )["input_ids"]

    prompt_ids = tokenizer.apply_chat_template(
        messages[:-1],
        tokenize=True,
        add_generation_prompt=True
    )["input_ids"]

    # Verify prompt is an exact prefix of the full conversation
    assert full_ids[:len(prompt_ids)] == prompt_ids

    # Keep the complete conversation
    assert len(full_ids) <= MAX_LENGTH

    # Mask system and user tokens; train only on assistant tokens
    labels = [-100] * len(prompt_ids) + full_ids[len(prompt_ids):]

    train_examples.append({
        "input_ids": full_ids,
        "attention_mask": [1] * len(full_ids),
        "labels": labels
    })

print("Prepared examples:", len(train_examples))
print("Maximum sequence length:",
      max(len(x["input_ids"]) for x in train_examples))

print("Assistant-only masking check:")
print("First example masked tokens:",
      sum(x == -100 for x in train_examples[0]["labels"]))
print("First example training tokens:",
      sum(x != -100 for x in train_examples[0]["labels"]))

print("All examples prepared successfully!")

Prepared examples: 1445
Maximum sequence length: 2215
Assistant-only masking check:
First example masked tokens: 133
First example training tokens: 82
All examples prepared successfully!


## 9. QLoRA Environment Setup and Diagnostics

These cells include quantization setup and troubleshooting performed while configuring the Colab environment.

In [ ]:
import gc
import torch

from transformers import (
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

from peft import (
    LoraConfig,
    prepare_model_for_kbit_training,
    get_peft_model
)

# Release the existing full-precision model
#del model
gc.collect()
torch.cuda.empty_cache()

# 4-bit quantization configuration
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

# Load quantized base model
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto"
)

# Prepare for QLoRA training
model = prepare_model_for_kbit_training(
    model,
    use_gradient_checkpointing=True
)

model.config.use_cache = False

# LoRA configuration
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj"
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)

model.print_trainable_parameters()

print("\nQLoRA model configured successfully!")
print("GPU memory allocated:",
      round(torch.cuda.memory_allocated() / 1024**3, 2), "GB")

ImportError: Using `bitsandbytes` 4-bit quantization requires bitsandbytes: `pip install -U bitsandbytes>=0.46.1`

In [ ]:
import bitsandbytes as bnb
print("BitsAndBytes version:", bnb.__version__)

BitsAndBytes version: 0.50.2


In [ ]:
import transformers.utils.import_utils as iu

print("BitsAndBytes detected:", iu.is_bitsandbytes_available())

BitsAndBytes detected: False


In [ ]:
del model

NameError: name 'model' is not defined

In [ ]:
!pip install -U bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 38.2 MB/s  0:00:01


## 10. Final QLoRA Training Pipeline

Reload the tokenizer and cleaned data, prepare assistant-only labels, configure the 4-bit LoRA adapter, and train the model.

In [ ]:
import json
from transformers import AutoTokenizer

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
MAX_LENGTH = 2304

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

with open("/content/train_clean.jsonl", "r", encoding="utf-8") as f:
    train_rows = [json.loads(line) for line in f if line.strip()]

with open("/content/dev.jsonl", "r", encoding="utf-8") as f:
    dev_rows = [json.loads(line) for line in f if line.strip()]

print("Tokenizer loaded")
print("Training records:", len(train_rows))
print("Dev records:", len(dev_rows))
print("Pad token:", tokenizer.pad_token)

Tokenizer loaded
Training records: 1445
Dev records: 200
Pad token: <|endoftext|>


In [ ]:
train_examples = []

for row in train_rows:
    messages = row["messages"]

    full_ids = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=False
    )["input_ids"]

    prompt_ids = tokenizer.apply_chat_template(
        messages[:-1],
        tokenize=True,
        add_generation_prompt=True
    )["input_ids"]

    assert full_ids[:len(prompt_ids)] == prompt_ids
    assert len(full_ids) <= MAX_LENGTH

    labels = [-100] * len(prompt_ids) + full_ids[len(prompt_ids):]

    train_examples.append({
        "input_ids": full_ids,
        "attention_mask": [1] * len(full_ids),
        "labels": labels
    })

print("Prepared examples:", len(train_examples))
print("Maximum sequence length:", max(len(x["input_ids"]) for x in train_examples))
print("Assistant-only masking:", all(
    all(v == -100 for v in x["labels"][:len(x["input_ids"]) - sum(v != -100 for v in x["labels"])])
    for x in train_examples
))

Prepared examples: 1445
Maximum sequence length: 2215
Assistant-only masking: True


In [ ]:
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import (
    LoraConfig,
    prepare_model_for_kbit_training,
    get_peft_model
)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto"
)

model = prepare_model_for_kbit_training(
    model,
    use_gradient_checkpointing=True
)

model.config.use_cache = False

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)

model.print_trainable_parameters()

print("QLoRA model configured successfully!")
print("GPU memory allocated:",
      round(torch.cuda.memory_allocated() / 1024**3, 2), "GB")

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

trainable params: 9,232,384 || all params: 1,552,946,688 || trainable%: 0.5945
QLoRA model configured successfully!
GPU memory allocated: 1.54 GB


In [ ]:
import torch
from torch.utils.data import Dataset

class TicketDataset(Dataset):
    def __init__(self, examples):
        self.examples = examples

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        return self.examples[idx]


train_dataset = TicketDataset(train_examples)

def collate_fn(features):
    max_len = max(len(x["input_ids"]) for x in features)

    input_ids = torch.full(
        (len(features), max_len),
        tokenizer.pad_token_id,
        dtype=torch.long
    )

    attention_mask = torch.zeros(
        (len(features), max_len),
        dtype=torch.long
    )

    labels = torch.full(
        (len(features), max_len),
        -100,
        dtype=torch.long
    )

    for i, item in enumerate(features):
        n = len(item["input_ids"])
        input_ids[i, :n] = torch.tensor(item["input_ids"])
        attention_mask[i, :n] = 1
        labels[i, :n] = torch.tensor(item["labels"])

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels
    }

print("Training dataset:", len(train_dataset))
print("Collator ready!")

Training dataset: 1445
Collator ready!


In [ ]:
import time
from transformers import Trainer, TrainingArguments

training_args = TrainingArguments(
    output_dir="/content/nimbuspay_qlora",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    num_train_epochs=2,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=20,
    save_strategy="epoch",
    gradient_checkpointing=True,
    optim="paged_adamw_8bit",
    warmup_steps=10,
    lr_scheduler_type="cosine",
    max_grad_norm=0.3,
    report_to="none",
    remove_unused_columns=False,
    dataloader_num_workers=0
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    data_collator=collate_fn
)

print("Training configuration ready!")
print("Epochs:", training_args.num_train_epochs)
print("Effective batch size:", 8)

Training configuration ready!
Epochs: 2
Effective batch size: 8


In [ ]:
import time
import torch

torch.cuda.reset_peak_memory_stats()

start_time = time.time()

train_result = trainer.train()

training_minutes = (time.time() - start_time) / 60
peak_gpu_gb = torch.cuda.max_memory_allocated() / 1024**3

print("\nTraining completed!")
print("Training time:", round(training_minutes, 2), "minutes")
print("Peak GPU memory:", round(peak_gpu_gb, 2), "GB")
print("Training loss:", train_result.training_loss)

Step,Training Loss
20,0.944649
40,0.079130
60,0.060202
80,0.041553
100,0.036248
120,0.035826
140,0.029073
160,0.022993
180,0.019648
200,0.011092



Training completed!
Training time: 32.34 minutes
Peak GPU memory: 6.18 GB
Training loss: 0.07614079596042551


In [ ]:
ADAPTER_PATH = "/content/nimbuspay_qlora_adapter"

model.save_pretrained(ADAPTER_PATH)
tokenizer.save_pretrained(ADAPTER_PATH)

print("Adapter saved successfully!")
print("Location:", ADAPTER_PATH)
print("Training time:", round(training_minutes, 2), "minutes")
print("Peak GPU memory:", round(peak_gpu_gb, 2), "GB")

Adapter saved successfully!
Location: /content/nimbuspay_qlora_adapter
Training time: 32.34 minutes
Peak GPU memory: 6.18 GB


## 11. Supplementary Checks

Inspect saved baseline outputs, the scoring script, and files available in the Colab runtime.

In [ ]:
print("Baseline 2 prediction sample:")
with open("/content/baseline2_predictions.jsonl", "r") as f:
    print(f.readline())

print("\nScoring script usage:")
!tail -60 /content/score.py

Baseline 2 prediction sample:


FileNotFoundError: [Errno 2] No such file or directory: '/content/baseline2_predictions.jsonl'

In [ ]:
!tail -80 /content/score.py


    python score.py --gold data/dev.jsonl --pred my_dev_predictions.jsonl

Prediction file: one JSON object per line
    {"id": "dv-00001", "output": "<the raw text your model generated>"}

Scoring is strict:
  * "output" must be ONLY a JSON object (leading/trailing whitespace is ignored).
  * Every field must match exactly, including type (true is not 1, 129900 is not "129900").
  * A missing id or invalid JSON counts as wrong on every field.
"""
import argparse, json
from collections import defaultdict

KEYS = ["category", "priority", "amount_paise", "txn_date", "txn_id", "channel", "language", "needs_human"]


def load(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]


def same(a, b):
    return type(a) is type(b) and a == b


def parse(text):
    try:
        obj = json.loads(text.strip())
    except (json.JSONDecodeError, AttributeError):
        return None
    return obj if isinstance(obj, dict) else None


def s

In [ ]:
import os
print(os.listdir("/content"))

['.config', 'nimbuspay_qlora', 'score.py', 'dev.jsonl', 'nimbuspay_qlora_adapter', 'train.jsonl', 'test_inputs.jsonl', 'train_clean.jsonl', 'sample_data']


## 12. Development-Set Evaluation

Generate predictions on the 200 labelled development examples and evaluate them with the provided scorer.

In [ ]:
import json
import torch
from tqdm.auto import tqdm

model.eval()
tokenizer.padding_side = "left"

dev_predictions = []
BATCH_SIZE = 4

for start in tqdm(range(0, len(dev_rows), BATCH_SIZE)):
    batch = dev_rows[start:start + BATCH_SIZE]

    conversations = [
        row["messages"][:-1] for row in batch
    ]

    inputs = tokenizer.apply_chat_template(
        conversations,
        tokenize=True,
        add_generation_prompt=True,
        padding=True,
        return_tensors="pt",
        return_dict=True
    )

    inputs = {k: v.to("cuda") for k, v in inputs.items()}

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=200,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id
        )

    generated_tokens = outputs[:, inputs["input_ids"].shape[1]:]

    for row, tokens in zip(batch, generated_tokens):
        raw_output = tokenizer.decode(
            tokens,
            skip_special_tokens=True
        )

        dev_predictions.append({
            "id": row["id"],
            "output": raw_output
        })

with open("/content/qlora_dev_predictions.jsonl", "w", encoding="utf-8") as f:
    for item in dev_predictions:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

print("Predictions generated:", len(dev_predictions))
print("Saved: /content/qlora_dev_predictions.jsonl")
print("\nFirst prediction:")
print(dev_predictions[0])

  0%|          | 0/50 [00:00<?, ?it/s]

Predictions generated: 200
Saved: /content/qlora_dev_predictions.jsonl

First prediction:
{'id': 'dv-00001', 'output': '{"category": "other", "priority": "P3", "amount_paise": null, "txn_date": null, "txn_id": null, "channel": null, "language": "en", "needs_human": false}'}


In [ ]:
!python /content/score.py \
  --gold /content/dev.jsonl \
  --pred /content/qlora_dev_predictions.jsonl

group                            n   valid   exact  fields
all                            200  100.0%   82.5%   96.9%
category=account_access         19  100.0%  100.0%  100.0%
category=fraud                  15  100.0%   73.3%   95.0%
category=kyc                    24  100.0%  100.0%  100.0%
category=offers                 18  100.0%   88.9%   98.6%
category=other                  16  100.0%  100.0%  100.0%
category=payment_failed         53  100.0%   73.6%   95.3%
category=refund                 55  100.0%   72.7%   95.2%

per-field accuracy (all rows)
  category        98.5%
  priority        86.5%
  amount_paise    97.5%
  txn_date        99.0%
  txn_id         100.0%
  channel         99.5%
  language       100.0%
  needs_human     94.5%


In [ ]:
import json
import subprocess

# Save current QLoRA evaluation metrics
with open("/content/qlora_clean_r8_lr2e4_e2_metrics.json", "w") as f:
    result = subprocess.run(
        [
            "python", "/content/score.py",
            "--gold", "/content/dev.jsonl",
            "--pred", "/content/qlora_dev_predictions.jsonl",
            "--json"
        ],
        capture_output=True,
        text=True,
        check=True
    )
    f.write(result.stdout)

# Load original, uncleaned training data
with open("/content/train.jsonl", "r", encoding="utf-8") as f:
    raw_rows = [json.loads(line) for line in f if line.strip()]

print("Raw training records:", len(raw_rows))
print("Clean training records:", len(train_rows))
print("Raw sample ID:", raw_rows[0]["id"])
print("Clean sample ID:", train_rows[0]["id"])
print("Metrics saved successfully!")

Raw training records: 1445
Clean training records: 1445
Raw sample ID: tr-00001
Clean sample ID: tr-00001
Metrics saved successfully!


## 13. Raw-Data Experiment Preparation

Prepare raw training examples for the raw-versus-clean experiment. This preparation is not itself a completed experiment.

In [ ]:
raw_lengths = []

for row in raw_rows:
    messages = row["messages"]

    ids = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=False
    )["input_ids"]

    raw_lengths.append(len(ids))

print("Raw records:", len(raw_lengths))
print("Maximum length:", max(raw_lengths))
print("Records over 2304:", sum(x > MAX_LENGTH for x in raw_lengths))
print("Records fitting 2304:", sum(x <= MAX_LENGTH for x in raw_lengths))
print("95th percentile:", sorted(raw_lengths)[int(0.95 * len(raw_lengths))])

Raw records: 1445
Maximum length: 2215
Records over 2304: 0
Records fitting 2304: 1445
95th percentile: 225


In [ ]:
raw_examples = []

for row in raw_rows:
    messages = row["messages"]

    full_ids = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=False
    )["input_ids"]

    prompt_ids = tokenizer.apply_chat_template(
        messages[:-1],
        tokenize=True,
        add_generation_prompt=True
    )["input_ids"]

    assert full_ids[:len(prompt_ids)] == prompt_ids
    assert len(full_ids) <= MAX_LENGTH

    labels = [-100] * len(prompt_ids) + full_ids[len(prompt_ids):]

    raw_examples.append({
        "input_ids": full_ids,
        "attention_mask": [1] * len(full_ids),
        "labels": labels
    })

print("Raw examples prepared:", len(raw_examples))
print("Maximum sequence length:", max(len(x["input_ids"]) for x in raw_examples))
print("Assistant-only masking:", all(
    all(v == -100 for v in x["labels"][:len(x["input_ids"]) - sum(v != -100 for v in x["labels"])])
    for x in raw_examples
))

Raw examples prepared: 1445
Maximum sequence length: 2215
Assistant-only masking: True


## 14. Hidden-Test Predictions

Generate and save raw model outputs for all 400 unlabelled test tickets.

In [ ]:
import json
import torch
from tqdm.auto import tqdm

with open("/content/test_inputs.jsonl", "r") as f:
    test_rows = [json.loads(line) for line in f if line.strip()]

model.eval()
tokenizer.padding_side = "left"

test_predictions = []
BATCH_SIZE = 4

for start in tqdm(range(0, len(test_rows), BATCH_SIZE)):
    batch = test_rows[start:start + BATCH_SIZE]
    conversations = [row["messages"] for row in batch]

    inputs = tokenizer.apply_chat_template(
        conversations,
        tokenize=True,
        add_generation_prompt=True,
        padding=True,
        return_tensors="pt",
        return_dict=True
    )

    inputs = {k: v.to("cuda") for k, v in inputs.items()}

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=200,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id
        )

    generated = outputs[:, inputs["input_ids"].shape[1]:]

    for row, tokens in zip(batch, generated):
        raw_output = tokenizer.decode(
            tokens, skip_special_tokens=True
        )
        test_predictions.append({
            "id": row["id"],
            "output": raw_output
        })

with open("/content/predictions_test.jsonl", "w") as f:
    for item in test_predictions:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

print("Test predictions:", len(test_predictions))
print("Saved: /content/predictions_test.jsonl")

  0%|          | 0/100 [00:00<?, ?it/s]

Test predictions: 400
Saved: /content/predictions_test.jsonl


In [ ]:
import json
import os

with open("/content/predictions_test.jsonl", "r") as f:
    final_preds = [json.loads(line) for line in f if line.strip()]

print("Total predictions:", len(final_preds))
print("Unique IDs:", len(set(x["id"] for x in final_preds)))
print("First prediction:")
print(json.dumps(final_preds[0], indent=2))

print("File size:", round(
    os.path.getsize("/content/predictions_test.jsonl") / 1024, 2
), "KB")

Total predictions: 400
Unique IDs: 400
First prediction:
{
  "id": "ts-00001",
  "output": "{\"category\": \"fraud\", \"priority\": \"P1\", \"amount_paise\": null, \"txn_date\": \"2026-07-00\", \"txn_id\": \"NP6090913363\", \"channel\": null, \"language\": \"hi-en\", \"needs_human\": true}"
}
File size: 87.66 KB


## 15. Error Analysis

Compare development predictions with gold labels, inspect incorrect examples, and summarize field-level errors.

In [ ]:
import json
import pandas as pd

FIELDS = [
    "category", "priority", "amount_paise", "txn_date",
    "txn_id", "channel", "language", "needs_human"
]

prediction_lookup = {x["id"]: x["output"] for x in dev_predictions}

errors = []

for row in dev_rows:
    ticket_id = row["id"]

    gold_raw = row["messages"][-1]["content"]
    pred_raw = prediction_lookup[ticket_id]

    try:
        gold = json.loads(gold_raw)
        pred = json.loads(pred_raw)

        mismatches = [
            field for field in FIELDS
            if gold.get(field) != pred.get(field)
        ]

        if mismatches:
            errors.append({
                "id": ticket_id,
                "ticket": row["messages"][1]["content"],
                "gold": gold,
                "prediction": pred,
                "wrong_fields": mismatches
            })

    except Exception as e:
        print("Parsing issue:", ticket_id, str(e))

print("Total dev errors:", len(errors))
print("Showing first 10 errors:\n")

for item in errors[:10]:
    print("=" * 80)
    print("ID:", item["id"])
    print("Wrong fields:", item["wrong_fields"])
    print("Ticket:", item["ticket"][:700])
    print("GOLD:", json.dumps(item["gold"], ensure_ascii=False))
    print("PRED:", json.dumps(item["prediction"], ensure_ascii=False))

Total dev errors: 35
Showing first 10 errors:

ID: dv-00005
Wrong fields: ['priority', 'needs_human']
Ticket: Received: 2026-08-09 (Sun)
Source: ivr-transcript

[IVR transcript] Hey, I tried to make a transfer but it failed. Dated Aug 1. Total 49680 rupees. Ref: NP9512428472 Kindly resolve this asap.
GOLD: {"category": "payment_failed", "priority": "P2", "amount_paise": 4968000, "txn_date": "2026-08-01", "txn_id": "NP9512428472", "channel": null, "language": "en", "needs_human": false}
PRED: {"category": "payment_failed", "priority": "P1", "amount_paise": 4968000, "txn_date": "2026-08-01", "txn_id": "NP9512428472", "channel": null, "language": "en", "needs_human": true}
ID: dv-00011
Wrong fields: ['priority', 'needs_human']
Ticket: Received: 2026-06-17 (Wed)
Source: email

Subject: Problem with NimbusPay

URGENT!!
Why am I missing my refund. App version is 4.2.7. ₹84,137 was the value. This happened yesterday. ID: np3333133441

Regards,
Pooja Nair
GOLD: {"category": "refund", "priority

In [ ]:
import json
import pandas as pd

error_rows = []

for item in errors:
    error_rows.append({
        "id": item["id"],
        "ticket": item["ticket"],
        "gold": json.dumps(item["gold"], ensure_ascii=False),
        "prediction": json.dumps(item["prediction"], ensure_ascii=False),
        "wrong_fields": ", ".join(item["wrong_fields"]),
        "num_wrong_fields": len(item["wrong_fields"])
    })

error_df = pd.DataFrame(error_rows)

error_df.to_csv(
    "/content/error_analysis.csv",
    index=False
)

# Count mistakes by field
field_errors = {}

for item in errors:
    for field in item["wrong_fields"]:
        field_errors[field] = field_errors.get(field, 0) + 1

print("Total incorrect dev records:", len(error_df))
print("\nErrors by field:")
print(pd.Series(field_errors).sort_values(ascending=False))

print("\nSaved: /content/error_analysis.csv")

Total incorrect dev records: 35

Errors by field:
priority        27
needs_human     11
amount_paise     5
category         3
txn_date         2
channel          1
dtype: int64

Saved: /content/error_analysis.csv


## 16. Experiment Results Summary

Record measured experiment results. At present, this table contains the clean-data QLoRA baseline only.

In [ ]:
import pandas as pd

experiment_results = pd.DataFrame([
    {
        "experiment": "Clean data - QLoRA baseline",
        "training_data": "Clean",
        "quantization": "4-bit NF4",
        "lora_rank": 8,
        "learning_rate": 0.0002,
        "epochs": 2,
        "exact_match": 82.5,
        "mean_field_accuracy": 96.9,
        "peak_gpu_memory_gb": 6.18,
        "training_minutes": 32.34
    }
])

experiment_results.to_csv(
    "/content/experiment_results.csv",
    index=False
)

print(experiment_results.to_string(index=False))
print("\nSaved: experiment_results.csv")

                 experiment training_data quantization  lora_rank  learning_rate  epochs  exact_match  mean_field_accuracy  peak_gpu_memory_gb  training_minutes
Clean data - QLoRA baseline         Clean    4-bit NF4          8         0.0002       2         82.5                 96.9                6.18             32.34

Saved: experiment_results.csv


In [ ]:
print("Raw examples:", len(raw_examples))
print("Clean examples:", len(train_examples))
print("Raw max length:", max(len(x["input_ids"]) for x in raw_examples))
print("Clean max length:", max(len(x["input_ids"]) for x in train_examples))

Experiment 1 – Raw vs clean training data
This cell trains on raw data with the same configuration as your original clean-data QLoRA baseline.

In [ ]:

# EXPERIMENT 1: RAW DATA vs CLEAN DATA
# Only the training dataset changes.
# Baseline: Clean data, 4-bit QLoRA, rank 8.

import os
import gc
import time
import torch
import pandas as pd

from transformers import (
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer
)
from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training
)

# Do not start training without a GPU
assert torch.cuda.is_available(), "GPU unavailable. Run this cell when GPU access returns."

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
OUTPUT_DIR = "/content/nimbuspay_raw_qlora"
ADAPTER_DIR = "/content/nimbuspay_raw_qlora_adapter"

# Dataset
raw_dataset = TicketDataset(raw_examples)

# 4-bit QLoRA
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

raw_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto"
)

raw_model = prepare_model_for_kbit_training(
    raw_model,
    use_gradient_checkpointing=True
)

raw_model.config.use_cache = False

raw_lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

raw_model = get_peft_model(raw_model, raw_lora_config)
raw_model.print_trainable_parameters()

# Same training settings as clean baseline
raw_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    num_train_epochs=2,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=20,
    save_strategy="epoch",
    gradient_checkpointing=True,
    optim="paged_adamw_8bit",
    warmup_steps=10,
    lr_scheduler_type="cosine",
    max_grad_norm=0.3,
    report_to="none",
    remove_unused_columns=False,
    dataloader_num_workers=0
)

raw_trainer = Trainer(
    model=raw_model,
    args=raw_args,
    train_dataset=raw_dataset,
    data_collator=collate_fn
)

start_time = time.time()
raw_trainer.train()
raw_training_minutes = (time.time() - start_time) / 60

peak_memory_gb = (
    torch.cuda.max_memory_allocated() / 1024**3
)

raw_model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

print("Raw-data training completed!")
print("Training minutes:", round(raw_training_minutes, 2))
print("Peak GPU memory (GB):", round(peak_memory_gb, 2))
print("Adapter saved:", ADAPTER_DIR)


Experiment 2 – FP16 LoRA vs 4-bit QLoRA

FP16 loading of the base model may require more VRAM, so we'll assess memory before running it.

In [ ]:

# EXPERIMENT 2: FP16 LoRA
# Compare against the existing 4-bit QLoRA baseline.

import torch
import time
import gc

assert torch.cuda.is_available(), "GPU unavailable."

fp16_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto"
)

fp16_model.config.use_cache = False

fp16_lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

fp16_model = get_peft_model(fp16_model, fp16_lora_config)

fp16_args = TrainingArguments(
    output_dir="/content/nimbuspay_fp16_lora",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    num_train_epochs=2,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=20,
    save_strategy="epoch",
    gradient_checkpointing=True,
    optim="adamw_torch",
    warmup_steps=10,
    lr_scheduler_type="cosine",
    max_grad_norm=0.3,
    report_to="none",
    remove_unused_columns=False,
    dataloader_num_workers=0
)

fp16_trainer = Trainer(
    model=fp16_model,
    args=fp16_args,
    train_dataset=train_dataset,
    data_collator=collate_fn
)

torch.cuda.reset_peak_memory_stats()
start_time = time.time()

fp16_trainer.train()

fp16_minutes = (time.time() - start_time) / 60
fp16_peak_memory = torch.cuda.max_memory_allocated() / 1024**3

fp16_model.save_pretrained("/content/nimbuspay_fp16_adapter")

print("FP16 LoRA training completed!")
print("Minutes:", round(fp16_minutes, 2))
print("Peak GPU memory:", round(fp16_peak_memory, 2), "GB")


Experiment 3 – LoRA rank 16

In [ ]:

# EXPERIMENT 3: RANK 16 vs RANK 8
# Only LoRA rank changes.

assert torch.cuda.is_available(), "GPU unavailable."

rank16_bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

rank16_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=rank16_bnb,
    device_map="auto"
)

rank16_model = prepare_model_for_kbit_training(
    rank16_model,
    use_gradient_checkpointing=True
)
rank16_model.config.use_cache = False

rank16_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

rank16_model = get_peft_model(rank16_model, rank16_config)

rank16_args = TrainingArguments(
    output_dir="/content/nimbuspay_rank16",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    num_train_epochs=2,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=20,
    save_strategy="epoch",
    gradient_checkpointing=True,
    optim="paged_adamw_8bit",
    warmup_steps=10,
    lr_scheduler_type="cosine",
    max_grad_norm=0.3,
    report_to="none",
    remove_unused_columns=False,
    dataloader_num_workers=0
)

rank16_trainer = Trainer(
    model=rank16_model,
    args=rank16_args,
    train_dataset=train_dataset,
    data_collator=collate_fn
)

torch.cuda.reset_peak_memory_stats()
start_time = time.time()

rank16_trainer.train()

rank16_minutes = (time.time() - start_time) / 60
rank16_peak_memory = torch.cuda.max_memory_allocated() / 1024**3

rank16_model.save_pretrained("/content/nimbuspay_rank16_adapter")

print("Rank 16 training completed!")
print("Minutes:", round(rank16_minutes, 2))
print("Peak GPU memory:", round(rank16_peak_memory, 2), "GB")


Experiment 4 – LoRA dropout 0.1

In [ ]:

# EXPERIMENT 4: DROPOUT 0.1 vs BASELINE DROPOUT 0.05
# Only dropout changes.

assert torch.cuda.is_available(), "GPU unavailable."

drop_bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

drop_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=drop_bnb,
    device_map="auto"
)

drop_model = prepare_model_for_kbit_training(
    drop_model,
    use_gradient_checkpointing=True
)
drop_model.config.use_cache = False

drop_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
    lora_dropout=0.1,
    bias="none",
    task_type="CAUSAL_LM"
)

drop_model = get_peft_model(drop_model, drop_config)

drop_args = TrainingArguments(
    output_dir="/content/nimbuspay_dropout01",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    num_train_epochs=2,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=20,
    save_strategy="epoch",
    gradient_checkpointing=True,
    optim="paged_adamw_8bit",
    warmup_steps=10,
    lr_scheduler_type="cosine",
    max_grad_norm=0.3,
    report_to="none",
    remove_unused_columns=False,
    dataloader_num_workers=0
)

drop_trainer = Trainer(
    model=drop_model,
    args=drop_args,
    train_dataset=train_dataset,
    data_collator=collate_fn
)

torch.cuda.reset_peak_memory_stats()
start_time = time.time()

drop_trainer.train()

drop_minutes = (time.time() - start_time) / 60
drop_peak_memory = torch.cuda.max_memory_allocated() / 1024**3

drop_model.save_pretrained("/content/nimbuspay_dropout01_adapter")

print("Dropout experiment completed!")
print("Minutes:", round(drop_minutes, 2))
print("Peak GPU memory:", round(drop_peak_memory, 2), "GB")
